# Q05: shift test-time augmentation (evaluation only)

NYU Tandon Deep Learning (Fall 2026), Project 1 ([Kaggle competition](https://www.kaggle.com/competitions/cifar-10-res-net-architecture-challenge-dl)).
Checklist task **Q05** (owner J.H.): does averaging a few small translations, in addition to the horizontal flip used
by every submission so far, change accuracy on the fixed internal 5k validation split? The notebook trains nothing.
It loads saved E checkpoints, rebuilds each model from the architecture options saved with it, and evaluates fixed
views of every validation image. Test images are forwarded only for the optional CSVs, one image at a time per view;
no statistic is computed from them. Normalization always comes from the checkpoint (training split only).

**Pre-declared on 2026-11-03, before any Q05 result was seen** (section 5 fixes them as version-locked constants):

| View set | Views | Window offsets (dy, dx) in pixels, each unflipped and horizontally flipped |
|---|---:|---|
| `flip2` | 2 | (0, 0): the current inference (original + flip) |
| `shift5x2` | 10 | (0, 0), (0, ±2), (±2, 0): centre, ±2 px horizontal, ±2 px vertical |
| `grid9x2` | 18 | the 3 × 3 grid {−2, 0, +2}²: shift5x2 plus the four diagonal offsets |

- **Shifts.** Each image is padded by 2 px with the run's own training crop padding (saved `crop_padding_mode`;
  runs saved before that switch used black borders) and the 32 × 32 window at the offset is cut out. `constant` is a
  zero (black) border in uint8 space before normalization, exactly like the training random crop (torchvision
  `RandomCrop(32, padding=4)` with fill 0 on the TrivialAugment/AutoAugment path, `F.pad` on the Cutout path);
  `reflect` mirrors without the edge pixel (T118). Every view is one of the windows the training random crop could
  produce. The horizontal flip is applied after the shift, as in training (crop, then flip).
- **Aggregation.** The mean of the views' softmax probabilities, exactly like the training notebooks' flip TTA
  (`predict_probs`, `val_accuracy`: `0.5 * (softmax(x) + softmax(flip(x)))`); no logit averaging is needed.
- **Comparisons.** For every model: accuracy of each view set, Δ versus `flip2`, and a paired exact McNemar test of
  `shift5x2` versus `flip2` and of `grid9x2` versus `flip2` (b = flip2 wrong & new right, c = flip2 right & new wrong,
  two-sided exact binomial p). All three sets are reported for every model; nothing is selected per model on
  validation, and p-values are unadjusted per-model diagnostics (about 13 models × 2 tests).
- **Sanity check.** `flip2` must reproduce each run's recorded accuracy for the same checkpoint and weights (e.g. T106
  final300 EMA+flip 97.48%); differences above 0.02 pp (one image) are flagged, and the plain single view and the
  stored per-image probabilities are compared too.
- **Precision.** fp32 without autocast, default (contiguous) memory format, batches of 1,000: the path
  (`load_model` + `predict_logits`) that produced every recorded headline. The checkpoint's training used bf16
  autocast, but its validation and prediction never did.

**Usage on Colab.** Upload this notebook, select an **A100 or L4 GPU**, keep the competition data/`cache.zip` in
`MyDrive/DL_Project1/` as for training, make sure H.W.'s shared folder is in MyDrive as the shortcut
`HW_DL_Project1` (E8–E11 live there), check `RUN_IDS` (section 1) and *Run all*. Approve the Drive mount. Each
evaluated model costs about 126,000 image forwards (18 views × 5,000 validation + 18 × 2,000 test images) plus reading a
61 MB checkpoint from Drive; by Q08's measured A100 throughput that is well under a minute of compute per model
(not measured for Q05; an L4 is slower).

**Outputs** (under `RESULT_DIR` only; extra roots are read-only; nothing is overwritten):
`result/reports/<MM-DD-HH-mm>_Q05_shift-tta-v1_summary.csv` and `..._report.json`; per run
`result/evaluations/validation/<RUN_ID>/<MM-DD-HH-mm>_<id>_q05-shift-tta-v1.json/.npz` (all metrics; float16
probabilities of the 18 views); and with `Q05_WRITE_TEST_CSVS=True`
`result/evaluations/test/<RUN_ID>/<MM-DD-HH-mm>_<id>_q05-shift5x2-submission.csv` and `..._q05-grid9x2-submission.csv`
with their `-probabilities.npz` and `-test-report.json`. These CSVs are **never submitted** by this notebook; every
Kaggle submission needs the user's approval.

CPU checks: `Archive/tests/test_q05_shift_tta.py`. Not yet run on a GPU; no Q05 result exists yet.

## 1. Configuration

Edit this cell, then Run all. `RUN_IDS` lists exact run IDs (legacy Drive names are translated as in the training
notebooks). Each run is evaluated at the checkpoint and weights of its recorded submission CSV (its prediction report);
a dict entry can override them. Checkpoints and their stored metadata are searched in `RESULT_DIR` first, then in each
read-only `Q05_EXTRA_RESULT_ROOTS` folder (on Colab a missing shortcut is awaited for `Q05_EXTRA_ROOT_WAIT_S` seconds,
then skipped with a warning). A run with no checkpoint in any root, with architecture options this notebook's E code
cannot rebuild (T108 head256, T109 multistage, IBN-a/IBN-b in T113 and T115–T117), or whose training has not completed
(T122 while it trains) is skipped with the reason printed; other problems (identity, split, integrity) are recorded as
failures and the next run is evaluated.

`VAL_SIZE`, `SPLIT_SEED` and `Q05_COMPETITION_CHECKS` must stay as given: the notebook refuses any other dataset or
split. The view sets themselves are not settings: they are fixed in section 5.

In [ ]:
# Q05 shift test-time augmentation: evaluation only, no training. Edit the values below, then Run all.
# The view sets, the 2 px offsets, the padding rule, the softmax averaging and the McNemar comparisons are fixed in
# section 5 (pre-declared 2026-11-03, before any Q05 result was seen); they are deliberately not settings here.

# Runs to evaluate: exact run IDs (legacy Drive names resolve automatically), or a dict
# {"run": <id>, "checkpoint": "best" | "last", "weights": "model" | "ema"} to override the protocol of the run's
# recorded submission CSV. Without an override each run is evaluated at the checkpoint/weights of its recorded
# submission (its *_prediction-report.json): last300/EMA for most runs, best/EMA for E1, E3, E8 and E9.
# Checkpoints and metadata are searched in RESULT_DIR first, then in each Q05_EXTRA_RESULT_ROOTS folder (read-only).
# Expected checkpoint locations: J.H.'s MyDrive/DL_Project1/result (RESULT_DIR on J.H.'s Colab) for E1-E7, E18, E19
# and T122; H.W.'s shared folder (MyDrive shortcut HW_DL_Project1) for E8-E13. A run found in no root is skipped.
RUN_IDS = [
    "10-30-21-05_E_pyramid-ver-1-T18-basic110-ema",       # E1/T18 Cutout; CSV best251 EMA+flip 97.22; Drive ...-ver-1-basic110-ema
    "10-31-19-19_E_pyramid-ver-2-T05-distill-ema",        # E2/T05 KD student (basic E); CSV last300 EMA+flip 97.06
    "10-31-20-06_E_pyramid-ver-3-T69-autoaug-only",       # E3/T69 AutoAugment; CSV best280 EMA+flip 97.18; Drive ...-T61-autoaug-only
    "11-01-00-07_E_pyramid-ver-4-T05-distill-autoaug",    # E4/T05 round 2 (basic E student); CSV last400 EMA+flip 97.18
    "11-01-00-10_E_pyramid-ver-5-T43-stochdepth-ema",     # E5/T43 DropPath 0.1; CSV last300 EMA+flip 96.96
    "11-01-20-08_E_pyramid-ver-6-T76-autoaug-cutout",     # E6/T76; CSV last300 EMA+flip 97.18
    "11-01-20-08_E_pyramid-ver-7-T77-autoaug-drop",       # E7/T77; CSV last300 EMA+flip 97.60
    "11-01-21-58_E_pyramid-ver-8-T88-trivialaug-ema",     # E8/T88 (H.W.); CSV best255 EMA+flip 97.08
    "11-01-23-46_E_pyramid-ver-9-T89-mixstyle-ema",       # E9/T89 (H.W.); MixStyle is the identity in eval; CSV best258 EMA+flip 97.08
    "11-02-03-08_E_pyramid-ver-10-T106-trivialaug-drop",  # E10/T106 (H.W.); CSV last300 EMA+flip 97.48
    "11-02-04-57_E_pyramid-ver-11-T107-ema9995-drop",     # E11/T107 (H.W.); CSV last300 EMA+flip 97.48
    "11-02-20-46_E_pyramid-ver-18-T118-reflect-ls005",    # E18/T118 reflect crop padding; CSV last300 EMA+flip 97.36
    "11-02-21-31_E_pyramid-ver-19-T119-cutmix-ema",       # E19/T119; CSV last300 EMA+flip 97.24
    # In progress: skipped (with a message) until training has completed, unless Q05_INCLUDE_INCOMPLETE = True.
    "11-03-02-04_E_pyramid-ver-20-T122-replica-d110",     # T122 bottleneck-110 alpha 128 + ShakeDrop 0.5, 1800 epochs
    # Architectures this notebook's E code cannot rebuild: listed so that the skip reason is printed.
    "11-02-06-43_E_pyramid-ver-12-T108-head256-drop",     # E12/T108 head256 head
    "11-02-08-31_E_pyramid-ver-13-T109-multistage-drop",  # E13/T109 multistage head
    "11-02-13-22_E_pyramid-ver-14-T115-ibn-a6s2",         # E14/T115 IBN-a (launch ID ...-ver-10-...)
    "11-02-13-23_E_pyramid-ver-15-T113-ibn-a6s1",         # E15/T113 IBN-a (launch ID ...-ver-11-...)
    "11-02-17-11_E_pyramid-ver-16-T116-ibnb2",            # E16/T116 IBN-b
    "11-02-17-11_E_pyramid-ver-17-T117-ibnb3",            # E17/T117 IBN-b + IN stem
    # Completed T87 SAM/ASAM exports exist under these IDs but are not registered in the checklist (their E8/E9
    # versions collide with T88/T89); uncomment to evaluate them as well.
    # "11-02-00-59_E_pyramid-ver-8-T87-sam",
    # "11-02-01-35_E_pyramid-ver-9-T87-asam",
]
# Read-only result folders of other accounts, searched after RESULT_DIR (as Q08_EXTRA_RESULT_ROOTS /
# KD_EXTRA_TEACHER_ROOTS): H.W.'s shared DL_Project1 folder added to MyDrive as the shortcut HW_DL_Project1.
Q05_EXTRA_RESULT_ROOTS = ["/content/drive/MyDrive/HW_DL_Project1/result"]
Q05_EXTRA_ROOT_WAIT_S = 60          # Colab only: a Drive shortcut can appear a little after mounting; wait this long
Q05_INCLUDE_INCOMPLETE = False      # True: also evaluate a run whose training has not completed (its current checkpoint)
Q05_WRITE_TEST_CSVS = True          # also write the shift5x2 (10-view) and grid9x2 (18-view) test CSVs of every
                                    # evaluated run; they are never submitted by this notebook (Kaggle needs approval)
Q05_SAVE_VIEW_PROBABILITIES = True  # per run: float16 validation softmax probabilities of all 18 views (.npz)
Q05_REPRO_TOLERANCE_PP = 0.02       # flip2 / plain must reproduce the stored accuracies within this (one image of 5,000);
                                    # larger differences are flagged (and that run's test CSVs are not written)
Q05_ALLOW_SHA256_MISMATCH = False   # a checkpoint whose SHA-256 differs from its recorded value is skipped unless True
Q05_STOP_ON_ERROR = False           # True: stop at the first failing run instead of recording it and continuing
Q05_SMOKE_N = 0                     # > 0: first N validation images only (quick check; no test CSVs; marked -smoke)

# Fixed split and data identity: keep these values (checked against the training notebooks' split)
VAL_SIZE = 5000
SPLIT_SEED = 0
Q05_COMPETITION_CHECKS = True  # require the competition cache fingerprint and the known split hashes

# Hardware (None = automatic). Evaluation is the training notebooks' val_accuracy / predict_probs path that produced
# every recorded headline: fp32 without autocast, default (contiguous) memory format, batches of 1,000 images.
DEVICE = None            # "cuda" | "mps" | "cpu"; None = $CIFAR_DEVICE, else cuda > mps > cpu
AMP = "off"              # not used for evaluation; kept for the shared environment cell
CHANNELS_LAST = False    # the headline path (load_model + predict_logits) evaluates contiguous tensors
CUDNN_BENCHMARK = None
PIN_MEMORY = None
WORKERS = None
EVAL_BATCH_SIZE = 1000   # same batch as the training notebooks' validation and prediction

# Runtime host, independent of the editor: auto | colab | kaggle | local.
# For a VS Code kernel connected to Colab, set "colab" if auto detection is ambiguous.
RUNTIME_ENV = "auto"     # explicit value overrides CIFAR_ENV; auto uses CIFAR_ENV then detection

# Paths (None = $CIFAR_DATA_DIR / $CIFAR_CACHE_DIR / $CIFAR_RESULT_DIR, else the default of the environment)
AUTO_DOWNLOAD = False      # Kaggle download/login code is commented out; use uploaded data
SAVE_CACHE_TO_DRIVE = True # on Colab, automatically save/verify MyDrive/DL_Project1/cache.zip
DATA_DIR = None            # Colab: MyDrive/DL_Project1/data/ (folder or override with a ZIP path)
CACHE_DIR = None
RESULT_DIR = None          # checkpoints are read from RESULT_DIR/checkpoints; Q05 outputs are written under it only
SHOW_SUMMARY = False       # the copied model cell prints no layer table here

# Baseline switch values for the verbatim model cell's own baseline build (section 4). Every checkpoint is rebuilt
# from its saved architecture options, whatever these values are.
DROP_PATH = 0.0
SHAKEDROP = 0.0
BLOCK = "basic"
DEPTH = 110
ALPHA = 98
ACTIVATION = "relu"

## 2. Environment, paths and device

The two code cells below are verbatim copies of `E_pyramid_t106_shakedrop.ipynb`'s environment cells (the regression
tests check this), so detection, Drive mounting and paths behave exactly as in training: Colab reads and writes
`MyDrive/DL_Project1/result` and keeps the cache in `/content/cache`; local runs use `project1/data`, `project1/cache`
and `project1/result`. On Colab select an A100 or L4 GPU.

In [ ]:
import copy
import csv
import hashlib
import importlib.util
import importlib.metadata
import io
import json
import math
import os
import platform
import random
import re
import shutil
import subprocess
import sys
import time
import tempfile
import zlib
import zipfile
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from datetime import datetime
from zoneinfo import ZoneInfo
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from matplotlib.ticker import MaxNLocator
from PIL import Image

ENVS = ("local", "colab", "kaggle")
KAGGLE_SLUG = "cifar-10-res-net-architecture-challenge-dl"
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/DL_Project1"
WORK_DIRS = {"colab": "/content", "kaggle": "/kaggle/working"}
COLAB_CKPT_EVERY = 5  # Drive writes are slow; best-ckpt.pt and the final epoch are always saved
PROJECT_DIR = Path.cwd().resolve()
LOCAL_PROJECT_DIR = PROJECT_DIR.parent if PROJECT_DIR.name == "Archive" else PROJECT_DIR
# Start method for the AutoAugment DataLoader workers: "fork" lets them use the Dataset class defined in this
# notebook. Under "spawn" (macOS, Windows) they cannot import it, so the pipeline runs in the main process there.
MP_CONTEXT = "fork" if sys.platform.startswith("linux") else None

_warned = set()


def _getenv(name):
    value = os.environ.get(name, "").strip()
    return value or None


def warn_once(msg):
    if msg not in _warned:
        _warned.add(msg)
        print(f"Warning: {msg}", flush=True)


def _module_exists(name):
    try:
        return importlib.util.find_spec(name) is not None
    except (ImportError, ValueError):
        return False


def detect_env():
    """Detect the kernel host, independent of the notebook editor (browser or VS Code)."""
    configured = str(globals().get("RUNTIME_ENV") or "auto").strip().lower()
    if configured not in ("auto",) + ENVS:
        raise ValueError(f"RUNTIME_ENV={configured!r}; expected auto or one of {ENVS}")
    requested = configured if configured != "auto" else (_getenv("CIFAR_ENV") or "auto").lower()
    if requested not in ("auto",) + ENVS:
        raise ValueError(f"CIFAR_ENV={requested!r}; expected auto or one of {ENVS}")
    if requested != "auto":
        return requested
    # Runtime markers beat directory names: /kaggle/input may also exist on Colab.
    if any(_getenv(key) for key in ("COLAB_RELEASE_TAG", "COLAB_BACKEND_VERSION")):
        return "colab"
    if _getenv("KAGGLE_KERNEL_RUN_TYPE"):
        return "kaggle"
    if _module_exists("google.colab"):
        return "colab"
    # A directory alone is weak evidence; use the full layout only as a fallback.
    if os.path.isdir("/kaggle/input") and os.path.isdir("/kaggle/working"):
        return "kaggle"
    return "local"


def work_dir(env=None):
    env = env or detect_env()
    return Path(_getenv("CIFAR_WORK_DIR") or WORK_DIRS.get(env, PROJECT_DIR))


def drive_dir():
    return Path(_getenv("CIFAR_DRIVE_DIR") or DRIVE_PROJECT_DIR)


def drive_mounted():
    return drive_dir().parent.is_dir()  # .../MyDrive exists once Drive is mounted


def kaggle_input_dir():
    return Path(_getenv("CIFAR_KAGGLE_INPUT") or "/kaggle/input")


def find_competition_data(root=None, max_depth=6):
    """Directories under `root` that contain train_labels.csv, best match first.

    Breadth-first, follows symlinks, skips *_images folders (50k files) and hidden folders.
    """
    root = Path(root or kaggle_input_dir())
    found, frontier = [], [(root, 0)]
    while frontier:
        d, depth = frontier.pop(0)
        try:
            entries = sorted(os.scandir(d), key=lambda e: e.name)
        except OSError:
            continue
        if any(e.name == "train_labels.csv" and e.is_file() for e in entries):
            found.append(d)
        if depth < max_depth:
            frontier += [(Path(e.path), depth + 1) for e in entries
                         if e.is_dir() and not e.name.endswith("_images") and not e.name.startswith(".")]

    def rank(d):
        complete = (d / "test.csv").is_file() and (d / "sample_submission.csv").is_file()
        return (not complete, KAGGLE_SLUG not in str(d), len(d.parts), str(d))

    return sorted(found, key=rank)


@dataclass
class Paths:
    env: str
    data_dir: Path
    cache_dir: Path
    result_dir: Path


def resolve_paths(data_dir=None, cache_dir=None, result_dir=None, env=None):
    """Data, cache and runs directories: argument > environment variable > environment default."""
    env = env or detect_env()
    data = data_dir or _getenv("CIFAR_DATA_DIR")
    cache = cache_dir or _getenv("CIFAR_CACHE_DIR")
    runs = result_dir or _getenv("CIFAR_RESULT_DIR")
    if env == "local":
        data = data or LOCAL_PROJECT_DIR / "data"
        cache = cache or LOCAL_PROJECT_DIR / "cache"
        runs = runs or LOCAL_PROJECT_DIR / "result"
    elif env == "colab":
        cache = cache or work_dir(env) / "cache"
        data = data or drive_dir() / "data"  # manually uploaded ZIPs or extracted competition files
        if runs is None:
            if drive_mounted():
                runs = drive_dir() / "result"
            else:
                runs = work_dir(env) / "result"
                warn_once(f"Google Drive is not mounted ({drive_dir().parent} missing); writing results to "
                          f"{runs}, which is lost when the runtime is recycled")
    else:  # kaggle
        cache = cache or work_dir(env) / "cache"
        runs = runs or work_dir(env) / "result"
        if data is None:
            found = find_competition_data()
            if found:
                data = found[0]
            else:
                data = work_dir(env) / "data"
                warn_once(f"no train_labels.csv found under {kaggle_input_dir()}; checking cache or manually supplied data")
    return Paths(env, Path(data), Path(cache), Path(runs))


def default_ckpt_every(env=None):
    return COLAB_CKPT_EVERY if (env or detect_env()) == "colab" else 1


def available_cpus():
    try:
        return len(os.sched_getaffinity(0))
    except AttributeError:  # macOS
        return os.cpu_count() or 1


def default_workers(device_type):
    """DataLoader workers (used by the per-image AutoAugment pipeline only)."""
    n = available_cpus()
    if device_type == "cuda":
        return min(8, max(2, n - 1))
    return min(4, max(1, n // 2))  # 4 on the 12-core M4 Pro, as before


def get_device(name=None):
    """`name` (DEVICE), else $CIFAR_DEVICE, else cuda > mps > cpu."""
    name = name or _getenv("CIFAR_DEVICE") or "auto"
    if name == "auto":
        if torch.cuda.is_available():
            return torch.device("cuda")
        if torch.backends.mps.is_available():
            return torch.device("mps")
        return torch.device("cpu")
    return torch.device(name)


def cuda_bf16_native(device):
    return torch.cuda.get_device_capability(device) >= (8, 0)  # Ampere or newer (L4, A100, ...)


def resolve_amp(requested, device):
    """'bf16', 'fp16' or 'off'. requested: auto | on | off | bf16 | fp16 (bools accepted)."""
    req = {True: "on", False: "off", None: "auto"}.get(requested, requested)
    req = str(req).lower()
    if req == "off":
        return "off"
    if device.type != "cuda":
        if req != "auto":
            # MPS: fp16 autocast gave no speed-up on an M4 Pro, and torch 2.5.1 GradScaler fails on MPS.
            print(f"Warning: AMP is only used on CUDA; training in fp32 on {device.type}.")
        return "off"
    if req == "fp16":
        return "fp16"
    if cuda_bf16_native(device):
        return "bf16"
    if req == "bf16":
        print("Warning: this GPU has no native bf16; using fp16 autocast + GradScaler instead.")
    return "fp16"


# torch.compile modes for the training step (README "Compilation"); a speed option, never part of the recipe.
COMPILE_MODES = ("off", "default", "reduce-overhead", "max-autotune", "max-autotune-no-cudagraphs")


def resolve_compile(requested, device, name="COMPILE"):
    """One of COMPILE_MODES. Compilation is used on CUDA only; elsewhere it warns and runs uncompiled ("off")."""
    if not isinstance(requested, str) or requested not in COMPILE_MODES:
        raise ValueError(f"{name} = {requested!r}; choose one of {COMPILE_MODES}")
    if requested != "off" and device.type != "cuda":
        print(f"Warning: {name}={requested!r} is only used on CUDA; running uncompiled on {device.type}.")
        return "off"
    return requested


def resolve_hardware(args, device):
    """Fill in the automatic (None) speed settings on `args` for `device`, in place."""
    cuda = device.type == "cuda"
    args.amp = resolve_amp(getattr(args, "amp", "auto"), device)
    args.compile = resolve_compile(getattr(args, "compile", "off"), device)
    if args.channels_last is None:
        args.channels_last = cuda
    elif args.channels_last and device.type == "mps":
        # torch 2.5.1 MPS: backward through conv/BN/ReLU in channels_last raises a stride error.
        print("Warning: CHANNELS_LAST is not supported on MPS with this PyTorch; disabling it.")
        args.channels_last = False
    if getattr(args, "cudnn_benchmark", None) is None:
        args.cudnn_benchmark = cuda
    if getattr(args, "pin_memory", None) is None:
        args.pin_memory = cuda
    if getattr(args, "workers", None) is None:
        args.workers = default_workers(device.type)
    if MP_CONTEXT is None:
        args.workers = 0  # see MP_CONTEXT
    return args


def device_name(device):
    if device.type == "cuda":
        props = torch.cuda.get_device_properties(device)
        return f"cuda ({props.name}, {props.total_memory / 2**30:.0f} GB)"
    if device.type == "cpu":
        return f"cpu ({available_cpus()} cores)"
    return f"{device.type} ({platform.machine()})"


def _on(x):
    return "on" if x else "off"


def summary_line(paths, device=None, args=None):
    """One line: environment, device, torch, speed settings and the output directories."""
    import torch

    parts = [f"env {paths.env}"]
    if device is not None:
        parts.append(device_name(device))
    parts.append(f"torch {torch.__version__}")
    if args is not None:
        parts.append(f"amp {args.amp} | channels_last {_on(args.channels_last)} | "
                     f"cudnn.benchmark {_on(args.cudnn_benchmark)} | workers {args.workers} | "
                     f"pin_memory {_on(args.pin_memory)}")
        if getattr(args, "compile", "off") != "off":
            parts.append(f"compile {args.compile}")
    parts.append(f"data {paths.data_dir} | cache {paths.cache_dir} | result {paths.result_dir}")
    return "[env] " + " | ".join(parts)

def run_timestamp():
    return datetime.now(ZoneInfo("America/New_York")).strftime("%m-%d-%H-%M")


# Exact historical aliases: checkpoint bytes and signed export manifests remain unchanged.
RUN_ID_ALIASES = {'10-30-08-40_A_resnet-ver-1-cutout-ema': '10-30-08-40_A_resnet-ver-1-T00-cutout-ema', '10-30-09-44_B_wrn-ver-1-cutout-ema': '10-30-09-44_B_wrn-ver-1-T00-cutout-ema', '10-30-09-46_C_seresnet-ver-1-cutout-ema': '10-30-09-46_C_seresnet-ver-1-T00-cutout-ema', '10-30-09-50_C_seresnet-ver-2-cutout-ema': '10-30-09-50_C_seresnet-ver-2-T00-cutout-ema', '10-30-19-41_B_wrn-ver-2-wide-ema': '10-30-19-41_B_wrn-ver-2-T08-wide-ema', '10-30-20-03_A_resnet-ver-6-eca-ema': '10-30-20-03_A_resnet-ver-2-T48-eca-ema', '10-30-20-04_C_seresnet-ver-8-se8-ema': '10-30-20-04_C_seresnet-ver-3-T06-se8-ema', '10-30-21-05_E_pyramid-ver-1-basic110-ema': '10-30-21-05_E_pyramid-ver-1-T18-basic110-ema', '10-30-21-07_G_resnext-ver-1-grouped-ema': '10-30-21-07_G_resnext-ver-1-T42-grouped-ema', '10-30-21-17_A_resnet-ver-7-mixup-ema': '10-30-21-17_A_resnet-ver-3-T49-mixup-ema', '10-30-21-18_B_wrn-ver-8-mixup-ema': '10-30-21-18_B_wrn-ver-3-T50-mixup-ema', '10-30-21-42_F_shake-ver-1-shake40': '10-30-21-42_F_shake-ver-1-T35-shake40', '10-30-21-47_C_seresnet-ver-31-mixup-ema': '10-30-21-47_C_seresnet-ver-4-T51-mixup-ema', '10-30-23-09_A_resnet-ver-4-early-ema': '10-30-23-09_A_resnet-ver-4-T22-early-ema', '10-30-23-10_B_wrn-ver-6-deep-ema': '10-30-23-10_B_wrn-ver-4-T24-deep-ema', '10-30-23-43_C_seresnet-ver-3-autoaug-cutout': '10-30-23-43_C_seresnet-ver-5-T01-autoaug-cutout', '10-30-23-56_A_resnet-ver-5-mid-ema': '10-30-23-56_A_resnet-ver-5-T26-mid-ema', '10-31-00-42_B_wrn-ver-5-balanced-ema': '10-31-00-42_B_wrn-ver-5-T23-balanced-ema', '10-31-00-50_C_seresnet-ver-19-reflect-ema': '10-31-00-50_C_seresnet-ver-6-T31-reflect-ema', '10-30-22-34_F_shake-ver-2-shake40-1800ep': '10-30-22-34_F_shake-ver-2-T35-shake40-1800ep', '10-31-20-06_E_pyramid-ver-3-T61-autoaug-only': '10-31-20-06_E_pyramid-ver-3-T69-autoaug-only'}


def canonical_run_id(run_id):
    return RUN_ID_ALIASES.get(run_id, run_id)


def run_task_id(run_id):
    match = re.search(r"-ver-\d+-(T\d{2,})-", canonical_run_id(run_id))
    return match.group(1) if match else "T00"


def run_paths(result_root, run_id, create=False):
    """One timestamped ID across directories grouped by artifact type."""
    if not run_id or Path(run_id).name != run_id or run_id in (".", ".."):
        raise ValueError("Run ID must be a single directory name")
    run_id = canonical_run_id(run_id)
    root = Path(result_root)
    folders = {"checkpoints": "checkpoints", "configs": "configs", "logs": "logs",
               "validation": "evaluations/validation", "test": "evaluations/test",
               "plots": "plots", "progress": "progress", "manifests": "manifests", "exports": "exports"}
    paths = SimpleNamespace(root=root, run_id=run_id,
                            **{key: root / folder / run_id for key, folder in folders.items()})
    if not paths.checkpoints.exists():
        for legacy, canonical in RUN_ID_ALIASES.items():
            old_dir = root / "checkpoints" / legacy
            if canonical == run_id and old_dir.is_dir():
                paths.checkpoints = old_dir
                break
    if create:
        for key in folders:
            getattr(paths, key).mkdir(parents=True, exist_ok=True)
    return paths


def paths_for_run(checkpoint_dir):
    checkpoint_dir = Path(checkpoint_dir)
    if checkpoint_dir.parent.name != "checkpoints":
        raise ValueError("Expected result/checkpoints/<RUN_ID>; legacy flat runs must be migrated explicitly")
    return run_paths(checkpoint_dir.parent.parent, checkpoint_dir.name)


def artifact_path(artifacts, category, role, extension, event=False, unique=False):
    """Stable training paths; dated evaluation/export paths with collision-safe role suffixes."""
    folder = getattr(artifacts, category)
    prefix = artifacts.run_id
    if event:
        identifier = artifacts.run_id.split('_', 1)[1] if '_' in artifacts.run_id else artifacts.run_id
        prefix = f"{run_timestamp()}_{identifier}"
    stem = f"{prefix}_{role}"
    path = folder / (stem + extension)
    if unique:
        index = 1
        while any(folder.glob(path.stem + '*')):
            index += 1
            path = folder / (f"{stem}-{index}" + extension)
    return path


def checkpoint_path(run_dir, which):
    if which not in ('best', 'last'):
        raise ValueError("Checkpoint role must be best or last")
    run_dir = Path(run_dir)
    canonical = canonical_run_id(run_dir.name)
    candidates = [run_dir.parent / canonical / f"{canonical}_{which}-ckpt.pt",
                  run_dir / f"{run_dir.name}_{which}-ckpt.pt"]
    for legacy, target in RUN_ID_ALIASES.items():
        if target == canonical:
            candidates.extend([run_dir.parent / canonical / f"{legacy}_{which}-ckpt.pt",
                               run_dir.parent / legacy / f"{legacy}_{which}-ckpt.pt"])
    return next((p for p in candidates if p.is_file()), candidates[0])


In [ ]:
ENV = detect_env()
print(f"[runtime] detected {ENV} | kernel {sys.executable} | cwd {Path.cwd()} | "
      f"RUNTIME_ENV={RUNTIME_ENV!r} | CIFAR_ENV={_getenv('CIFAR_ENV')!r}")
if ENV == "colab":
    try:
        from google.colab import drive
    except ImportError:  # CIFAR_ENV=colab outside Google Colab (testing)
        print("google.colab is not available: skipping drive.mount")
    else:
        drive.mount("/content/drive")

PATHS = resolve_paths(DATA_DIR, CACHE_DIR, RESULT_DIR, ENV)
DEV = get_device(DEVICE)
if ENV == "colab" and DEV.type != "cuda" and (DEVICE or _getenv("CIFAR_DEVICE")) != "cpu":
    raise RuntimeError("No GPU: choose Runtime > Change runtime type > A100 GPU, then Runtime > Run all again")
HW = resolve_hardware(SimpleNamespace(amp=AMP, channels_last=CHANNELS_LAST, cudnn_benchmark=CUDNN_BENCHMARK,
                                      pin_memory=PIN_MEMORY, workers=WORKERS), DEV)
print(summary_line(PATHS, DEV, HW))
print(f"python {platform.python_version()} | torchvision {torchvision.__version__} | numpy {np.__version__} | "
      f"{platform.platform()} | {available_cpus()} CPUs")
if ENV == "colab":
    print(f"Google Drive folder {drive_dir()}: {'found' if drive_dir().is_dir() else 'NOT FOUND'}")
if ENV == "kaggle":
    print(f"Kaggle input {kaggle_input_dir()}: train_labels.csv in "
          f"{[str(d) for d in find_competition_data()] or 'none'}")
if DEV.type == "cuda" and shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

## 3. Data and the shared validation split

The data helpers and the data cell are verbatim copies of the training notebook's cells: a verified local cache is
reused, then a compatible Drive `cache.zip`, then the manually uploaded competition data. The next cell holds verbatim
copies of the split, crop-padding, inference and checkpoint helpers this notebook needs (checked by the tests). The
split cell recomputes the 45k/5k split with `stratified_split`, checks the cache fingerprint (`42749a486a13…`) and the
SHA-256 of the validation indices and labels shared by every saved validation export, recomputes the training-split
normalization for comparison, and loads the test images for the optional CSVs. Each checkpoint's saved `train_idx` and
`val_idx` must equal this split exactly.

In [ ]:
NUM_CLASSES = 10
CLASSES = ("airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck")
CACHE_FORMAT = 2
SOURCE_CSVS = ("train_labels.csv", "test.csv", "sample_submission.csv")
CACHE_ARRAYS = ("train_images.npy", "train_labels.npy", "train_ids.npy", "test_images.npy", "test_ids.npy")


def sha256_file(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def _decode_pngs(paths):
    def load(p):
        with Image.open(p) as im:
            return np.asarray(im.convert("RGB"), dtype=np.uint8)

    with ThreadPoolExecutor(max_workers=min(8, os.cpu_count() or 1)) as pool:
        return np.stack(list(pool.map(load, paths, chunksize=256)))


def _save_npy(path, arr):
    tmp = path.with_suffix(".tmp")
    with open(tmp, "wb") as f:
        np.save(f, arr)
    os.replace(tmp, path)


def _copy_csvs(data_dir, cache_dir):
    if data_dir.resolve() == cache_dir.resolve():
        return
    for name in SOURCE_CSVS:
        tmp = cache_dir / (name + ".tmp")
        shutil.copyfile(data_dir / name, tmp)
        os.replace(tmp, cache_dir / name)


def _write_meta(cache_dir, n_train, n_test, shape, note):
    files = {name: {"sha256": sha256_file(cache_dir / name), "bytes": (cache_dir / name).stat().st_size}
             for name in CACHE_ARRAYS + SOURCE_CSVS}
    fingerprint = hashlib.sha256("".join(f"{n} {files[n]['sha256']}\n" for n in sorted(files)).encode())
    meta = {"format": CACHE_FORMAT, "n_train": n_train, "n_test": n_test, "shape": list(shape),
            "fingerprint": fingerprint.hexdigest(), "files": files, "note": note,
            "written": datetime.now().isoformat(timespec="seconds")}
    tmp = cache_dir / "meta.json.tmp"
    tmp.write_text(json.dumps(meta, indent=2))
    os.replace(tmp, cache_dir / "meta.json")  # written last: a cache without it is incomplete
    return meta


def read_meta(cache_dir):
    path = Path(cache_dir) / "meta.json"
    return json.loads(path.read_text()) if path.exists() else None


def build_cache(data_dir, cache_dir):
    """Decode all PNGs once into uint8 arrays (N, 32, 32, 3) and save them with the CSVs and meta.json."""
    data_dir, cache_dir = Path(data_dir), Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)

    train = pd.read_csv(data_dir / "train_labels.csv")
    print(f"[cache] decoding {len(train):,} training PNGs ...", flush=True)
    x = _decode_pngs([data_dir / "train_images" / f for f in train["filename"]])
    _save_npy(cache_dir / "train_images.npy", x)
    _save_npy(cache_dir / "train_labels.npy", train["label"].to_numpy(np.int64))
    _save_npy(cache_dir / "train_ids.npy", train["id"].to_numpy(np.int64))

    test = pd.read_csv(data_dir / "test.csv")
    print(f"[cache] decoding {len(test):,} test PNGs ...", flush=True)
    x = _decode_pngs([data_dir / "test_images" / f for f in test["filename"]])
    _save_npy(cache_dir / "test_images.npy", x)
    _save_npy(cache_dir / "test_ids.npy", test["id"].to_numpy(np.int64))

    _copy_csvs(data_dir, cache_dir)
    validate_cache_arrays(cache_dir)
    meta = _write_meta(cache_dir, len(train), len(test), x.shape[1:], f"built from {data_dir}")
    print(f"[cache] wrote {cache_dir} (fingerprint {meta['fingerprint'][:12]})", flush=True)
    return meta


def _upgrade_format1(data_dir, cache_dir):
    """Add CSV copies and hashes to a format-1 cache if its arrays match the CSVs. Returns success."""
    train = pd.read_csv(data_dir / "train_labels.csv")
    test = pd.read_csv(data_dir / "test.csv")
    try:
        x_train = np.load(cache_dir / "train_images.npy", mmap_mode="r")
        x_test = np.load(cache_dir / "test_images.npy", mmap_mode="r")
        ok = (x_train.shape[0] == len(train) and x_test.shape[0] == len(test)
              and x_train.shape[1:] == x_test.shape[1:]
              and np.array_equal(np.load(cache_dir / "train_labels.npy"), train["label"].to_numpy(np.int64))
              and np.array_equal(np.load(cache_dir / "train_ids.npy"), train["id"].to_numpy(np.int64))
              and np.array_equal(np.load(cache_dir / "test_ids.npy"), test["id"].to_numpy(np.int64)))
    except (OSError, ValueError):
        ok = False
    if not ok:
        return False
    shape = x_train.shape[1:]
    del x_train, x_test
    _copy_csvs(data_dir, cache_dir)
    validate_cache_arrays(cache_dir)
    meta = _write_meta(cache_dir, len(train), len(test), shape, f"format 1 cache upgraded with {data_dir}")
    print(f"[cache] upgraded {cache_dir} to format {CACHE_FORMAT} (arrays unchanged, "
          f"fingerprint {meta['fingerprint'][:12]})", flush=True)
    return True


def _cache_problem(meta, cache_dir, data_dir, has_csvs):
    """None if the format-2 cache is complete and matches the CSVs in data_dir, else the reason."""
    for name, info in meta["files"].items():
        path = cache_dir / name
        if not path.exists() or path.stat().st_size != info["bytes"]:
            return f"{name} is missing or has the wrong size"
    if has_csvs and data_dir.resolve() != cache_dir.resolve():
        for name in SOURCE_CSVS:
            if sha256_file(data_dir / name) != meta["files"][name]["sha256"]:
                return f"{data_dir / name} differs from the cached copy"
    return None


def ensure_cache(data_dir, cache_dir):
    """Return the cache dir, building, upgrading or rebuilding the cache only when needed.

    A complete format-2 cache is used as is; if data_dir also holds the CSVs, they must match the
    cached copies (otherwise the cache is rebuilt from the PNGs).
    """
    data_dir, cache_dir = Path(data_dir), Path(cache_dir)
    has_csvs = all((data_dir / n).is_file() for n in SOURCE_CSVS)
    has_pngs = has_csvs and (data_dir / "train_images").is_dir() and (data_dir / "test_images").is_dir()
    meta = read_meta(cache_dir)
    if meta is not None and meta.get("format") == CACHE_FORMAT:
        problem = _cache_problem(meta, cache_dir, data_dir, has_csvs)
        if problem is None:
            return cache_dir
        print(f"[cache] {problem}; rebuilding", flush=True)
    elif meta is not None and has_csvs and all((cache_dir / n).exists() for n in CACHE_ARRAYS):
        if _upgrade_format1(data_dir, cache_dir):
            return cache_dir
        print("[cache] format 1 cache does not match the CSVs; rebuilding", flush=True)
    if not has_pngs and meta is not None and meta.get("format") is None:
        raise FileNotFoundError(
            f"{cache_dir} is a format-1 cache and {data_dir} has no CSVs to upgrade it with; "
            f"set DATA_DIR to the folder with the competition CSVs")
    if not has_pngs:
        raise FileNotFoundError(
            f"no usable cache in {cache_dir} and no PNG images in {data_dir} to build one; set DATA_DIR "
            f"(folder with train_labels.csv, train_images/, ...) or CACHE_DIR, or on Colab upload cache.zip "
            f"to {drive_dir()}")
    build_cache(data_dir, cache_dir)
    return cache_dir


def verify_cache(cache_dir):
    """Check every cached file against the sha256 in meta.json; return meta (raises on a mismatch)."""
    cache_dir = Path(cache_dir)
    meta = read_meta(cache_dir)
    _check_cache_manifest(meta)
    bad = [n for n, info in meta["files"].items()
           if not (cache_dir / n).exists() or sha256_file(cache_dir / n) != info["sha256"]]
    if bad:
        raise RuntimeError(f"cache files do not match meta.json: {bad}")
    n_train, n_test = validate_cache_arrays(cache_dir)
    if (meta["n_train"], meta["n_test"], meta["shape"]) != (n_train, n_test, [32, 32, 3]):
        raise RuntimeError("Cache metadata does not match its arrays")
    return meta


def validate_cache_arrays(cache_dir):
    """Validate shapes, labels, IDs, and the correspondence between arrays and CSVs."""
    cache_dir = Path(cache_dir)
    frames = {name: pd.read_csv(cache_dir / name) for name in SOURCE_CSVS}
    for name, columns in (("train_labels.csv", ("id", "filename", "label")),
                          ("test.csv", ("id", "filename")),
                          ("sample_submission.csv", ("id", "label"))):
        frame = frames[name]
        if not set(columns).issubset(frame.columns) or frame.empty:
            raise ValueError(f"{name}: missing columns or no rows")
        if not pd.api.types.is_integer_dtype(frame["id"]) or not frame["id"].is_unique:
            raise ValueError(f"{name}: IDs must be unique integers")
        if "filename" in frame and (frame["filename"].isna().any() or not frame["filename"].is_unique):
            raise ValueError(f"{name}: filenames must be present and unique")
    train, test = frames["train_labels.csv"], frames["test.csv"]
    if not pd.api.types.is_integer_dtype(train["label"]) or not train["label"].between(0, 9).all():
        raise ValueError("Training labels must be integers in 0..9")
    for split, frame in (("train", train), ("test", test)):
        images = np.load(cache_dir / f"{split}_images.npy", mmap_mode="r", allow_pickle=False)
        ids = np.load(cache_dir / f"{split}_ids.npy", allow_pickle=False)
        if images.dtype != np.uint8 or images.shape != (len(frame), 32, 32, 3):
            raise ValueError(f"{split} images must be uint8 NHWC with shape (N,32,32,3)")
        if not np.array_equal(ids, frame["id"].to_numpy()):
            raise ValueError(f"{split} array IDs differ from CSV order")
    labels = np.load(cache_dir / "train_labels.npy", allow_pickle=False)
    if not np.issubdtype(labels.dtype, np.integer) or not np.array_equal(labels, train["label"].to_numpy()):
        raise ValueError("Cached labels differ from training CSV")
    if set(frames["sample_submission.csv"]["id"]) != set(test["id"]):
        raise ValueError("Sample submission and test IDs differ")
    return len(train), len(test)


def cache_fingerprint(cache_dir):
    meta = read_meta(cache_dir) or {}
    return meta.get("fingerprint")


def load_train(data_dir, cache_dir, mmap=False):
    cache_dir = ensure_cache(data_dir, cache_dir)
    images = np.load(cache_dir / "train_images.npy", mmap_mode="r" if mmap else None)
    return images, np.load(cache_dir / "train_labels.npy"), np.load(cache_dir / "train_ids.npy")


def load_test(data_dir, cache_dir):
    cache_dir = ensure_cache(data_dir, cache_dir)
    return np.load(cache_dir / "test_images.npy"), np.load(cache_dir / "test_ids.npy")


def _cache_matches_source(meta, source_dir):
    """An explicitly supplied set of CSVs must agree with a reused cache."""
    source_dir = Path(source_dir)
    return all(not (source_dir / name).is_file() or
               sha256_file(source_dir / name) == meta['files'][name]['sha256'] for name in SOURCE_CSVS)


def _check_cache_manifest(meta):
    required = set(CACHE_ARRAYS + SOURCE_CSVS)
    if (not isinstance(meta, dict) or meta.get('format') != CACHE_FORMAT
            or not isinstance(meta.get('files'), dict) or set(meta['files']) != required):
        raise ValueError('Cache needs a format-2 manifest with all required files')
    for info in meta['files'].values():
        if (not isinstance(info, dict) or not isinstance(info.get('bytes'), int)
                or info['bytes'] <= 0 or not isinstance(info.get('sha256'), str)
                or not re.fullmatch(r'[0-9a-f]{64}', info['sha256'])):
            raise ValueError('Invalid cache file size or SHA256 in manifest')
    fingerprint = hashlib.sha256(''.join(
        f"{name} {meta['files'][name]['sha256']}\n" for name in sorted(required)).encode()).hexdigest()
    if fingerprint != meta.get('fingerprint'):
        raise ValueError('Cache ZIP fingerprint differs from its manifest')
    return required


def verify_cache_zip(path):
    """Check ZIP contents and every uncompressed file hash, not just the ZIP header."""
    with zipfile.ZipFile(path) as z:
        names = z.namelist()
        if len(names) != len(set(names)) or set(names) != set(CACHE_ARRAYS + SOURCE_CSVS + ('meta.json',)):
            raise ValueError('Cache ZIP must contain exactly the required files at its root')
        meta = json.loads(z.read('meta.json'))
        for name in _check_cache_manifest(meta):
            info = meta['files'][name]
            digest = hashlib.sha256()
            with z.open(name) as f:
                for block in iter(lambda: f.read(1 << 22), b''):
                    digest.update(block)
            if z.getinfo(name).file_size != info['bytes'] or digest.hexdigest() != info['sha256']:
                raise ValueError(f'Cache ZIP checksum mismatch: {name}')
    return meta


def extract_zip_checked(archive, destination):
    """Extract inside a staging folder, rejecting escaping paths and symlink entries."""
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with zipfile.ZipFile(archive) as z:
        for member in z.infolist():
            target = (destination / member.filename).resolve()
            if ('\\' in member.filename or not target.is_relative_to(root)
                    or (member.external_attr >> 16) & 0o170000 == 0o120000):
                raise ValueError(f'Unsafe ZIP member: {member.filename}')
        z.extractall(destination)  # decompression also checks each member's CRC


def publish_data_directory(staged, destination):
    """Keep the previous directory until the validated replacement can be published."""
    staged, destination = Path(staged), Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    previous = destination.with_name(f'.{destination.name}.previous-{time.time_ns()}')
    if destination.exists():
        os.replace(destination, previous)
    try:
        os.replace(staged, destination)
    except BaseException:
        if previous.exists():
            try:
                os.replace(previous, destination)
            except OSError:
                # Keep recovery files even if the filesystem also refuses rollback.
                print(f'[data] previous files preserved at {previous}; restore them before retrying')
        raise
    else:
        if previous.is_dir() and not previous.is_symlink():
            shutil.rmtree(previous)
        else:
            previous.unlink(missing_ok=True)


def drive_cache_candidates(folder):
    folder = Path(folder)
    if not folder.is_dir():
        return []
    pattern = re.compile(r'cache(?: \(\d+\))?\.zip$')
    return sorted((p for p in folder.iterdir() if p.is_file() and pattern.fullmatch(p.name)),
                  key=lambda p: (p.stat().st_mtime_ns, p.name), reverse=True)


def fetch_drive_cache(folder, cache_dir, source_dir=None):
    """Try Drive cache copies newest first; never replace a cache with an unverified ZIP."""
    cache_dir = Path(cache_dir)
    cache_dir.parent.mkdir(parents=True, exist_ok=True)
    for archive in drive_cache_candidates(folder):
        try:
            with tempfile.TemporaryDirectory(prefix='.restore-cache-', dir=cache_dir.parent) as tmp:
                local_zip, staged = Path(tmp) / 'cache.zip', Path(tmp) / 'cache'
                shutil.copyfile(archive, local_zip)
                meta = verify_cache_zip(local_zip)
                if source_dir is not None and not _cache_matches_source(meta, source_dir):
                    print(f'[cache] skipping {archive.name}: source CSVs differ')
                    continue
                extract_zip_checked(local_zip, staged)
                verify_cache(staged)
                publish_data_directory(staged, cache_dir)
            print(f'[cache] restored {archive} -> {cache_dir}')
            return cache_dir
        except (OSError, ValueError, KeyError, RuntimeError, zipfile.BadZipFile, zlib.error, EOFError) as exc:
            print(f'[cache] cannot reuse {archive.name}: {type(exc).__name__}; trying another source')
    raise FileNotFoundError('No valid compatible Drive cache was found')


def save_drive_cache(cache_dir, folder):
    """Publish a verified cache.zip once; skip writing an identical valid backup."""
    cache_dir, folder = Path(cache_dir), Path(folder)
    meta = verify_cache(cache_dir)
    folder.mkdir(parents=True, exist_ok=True)
    destination = folder / 'cache.zip'
    if destination.is_file():
        try:
            if verify_cache_zip(destination)['fingerprint'] == meta['fingerprint']:
                print(f'[cache] Drive backup already verified: {destination}')
                return destination
        except (OSError, ValueError, KeyError, RuntimeError, zipfile.BadZipFile, zlib.error, EOFError):
            print('[cache] replacing an invalid Drive backup after verifying its replacement')
    partial = folder / f'.cache-{time.time_ns()}.zip.tmp'
    try:
        with tempfile.TemporaryDirectory(prefix='.export-cache-', dir=cache_dir.parent) as tmp:
            archive = Path(tmp) / 'cache.zip'
            with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED, compresslevel=1) as z:
                for name in CACHE_ARRAYS + SOURCE_CSVS + ('meta.json',):
                    z.write(cache_dir / name, arcname=name)
            verify_cache_zip(archive)
            shutil.copyfile(archive, partial)
            verify_cache_zip(partial)  # check the Drive copy before replacing the old backup
            os.replace(partial, destination)
    finally:
        partial.unlink(missing_ok=True)
    print(f'[cache] saved reusable Drive backup: {destination}')
    return destination


def find_raw_data(root, require_images=False):
    """Find CSVs under a wrapper folder, optionally checking every referenced PNG exists."""
    for folder in find_competition_data(root):
        if not all((folder / name).is_file() for name in SOURCE_CSVS):
            continue
        if not require_images:
            return folder
        complete = True
        for split, csv_name in (('train', 'train_labels.csv'), ('test', 'test.csv')):
            frame = pd.read_csv(folder / csv_name)
            if 'filename' not in frame or frame.empty or frame['filename'].isna().any():
                complete = False
                break
            image_root = (folder / f'{split}_images').resolve()
            for filename in frame['filename']:
                image = (image_root / str(filename)).resolve()
                if not image.is_relative_to(image_root) or not image.is_file():
                    complete = False
                    break
            if not complete:
                break
        if complete:
            return folder
    return None


# Kaggle download and login disabled: use manually uploaded data instead.
# def run_kaggle_download():
#     """Use KaggleHub in this notebook process and honor its returned cache/input path."""
#     if not _getenv('KAGGLE_CONFIG_DIR') and (Path('/content') / 'kaggle.json').is_file():
#         os.environ['KAGGLE_CONFIG_DIR'] = '/content'
#     try:
#         kagglehub = importlib.import_module('kagglehub')
#     except ModuleNotFoundError as exc:
#         if exc.name != 'kagglehub':
#             raise RuntimeError('A KaggleHub dependency is missing. Run %pip install -U "kagglehub>=1,<2", '
#                                'restart the kernel, and rerun from the configuration cell.') from None
#         print('[download] installing KaggleHub in this kernel environment')
#         installed = subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
#                                     '--disable-pip-version-check', 'kagglehub>=1,<2'],
#                                    capture_output=True, text=True)
#         if installed.returncode:
#             raise RuntimeError('Cannot install KaggleHub. Check internet access and use Python 3.11+; '
#                                'alternatively provide extracted data or a saved cache.')
#         importlib.invalidate_caches()
#         kagglehub = importlib.import_module('kagglehub')
#     print(f'[download] KaggleHub competition_download({KAGGLE_SLUG!r})', flush=True)
#     try:
#         # Do not assume a ZIP filename or pass newer optional API arguments.
#         # This also preserves credentials set by kagglehub.login() in this kernel.
#         downloaded = Path(kagglehub.competition_download(KAGGLE_SLUG))
#     except Exception as exc:
#         response = getattr(exc, 'response', None)
#         status = getattr(response, 'status_code', None)
#         if not isinstance(status, int):
#             status = getattr(exc, 'error_code', None)
#         status_text = f', HTTP {status}' if isinstance(status, int) and 100 <= status <= 599 else ''
#         kind = type(exc).__name__
#         if status == 401 or kind in ('UnauthenticatedError', 'CredentialError'):
#             print('[auth] Kaggle login is needed for this download. Enter your own API token in the login prompt.')
#             try:
#                 kagglehub.login()
#             except Exception as login_error:
#                 raise RuntimeError(
#                     f'Cannot open/complete Kaggle login ({type(login_error).__name__}). '
#                     'For notebook widgets, install ipywidgets and enable widget support in your editor, '
#                     'or use a masked getpass prompt to set KAGGLE_API_TOKEN (see README.md). '
#                     'Then rerun this data cell.') from None
#             # Notebook login displays an asynchronous widget. The kernel must become idle
#             # for its Login button to work; do not block/poll or download again immediately.
#             raise RuntimeError(
#                 f'Data setup paused for Kaggle login ({kind}{status_text}). '
#                 'Complete the login prompt above, then rerun THIS DATA CELL and continue the remaining cells. '
#                 'Do not restart the kernel after login. Existing data and valid caches need no login.') from None
#         if status == 403:
#             advice = ('Your account may need competition access or rule acceptance. '
#                       'If you need to switch credentials, use kagglehub.login() or set KAGGLE_API_TOKEN. '
#                       f'Confirm competition access and accepted rules at https://www.kaggle.com/competitions/{KAGGLE_SLUG}/rules.')
#         elif status == 404:
#             advice = f'Check the competition name {KAGGLE_SLUG!r} and that your account can access it.'
#         elif status == 429:
#             advice = 'Kaggle rate-limited the request. Wait before retrying.'
#         elif kind in ('ConnectionError', 'ConnectTimeout', 'ReadTimeout', 'Timeout'):
#             advice = 'Check the runtime internet connection and retry the data cell.'
#         elif isinstance(exc, OSError):
#             advice = 'Check local free space and write permissions for the KaggleHub cache.'
#         else:
#             advice = ('Check KaggleHub installation and account access; use kagglehub.login() or '
#                       'KAGGLE_API_TOKEN if authentication is required.')
#         # Keep useful diagnostics without echoing request URLs/tokens from exception text.
#         raise RuntimeError(f'KaggleHub download failed ({kind}{status_text}). {advice} '
#                            'Rerun the data cell after correcting the issue.') from None
#     if not downloaded.exists():
#         raise FileNotFoundError(f'KaggleHub returned a path that does not exist: {downloaded}')
#     print(f'[download] KaggleHub returned: {downloaded}')
#     return downloaded
#
#
# def download_competition_data(data_dir):
#     """Normalize a KaggleHub folder/ZIP or a legacy downloaded ZIP into the raw data directory."""
#     data_dir = Path(data_dir)
#     data_dir.parent.mkdir(parents=True, exist_ok=True)
#     legacy_archive = data_dir.parent / 'downloads' / KAGGLE_SLUG / f'{KAGGLE_SLUG}.zip'
#     downloaded = None
#     if legacy_archive.is_file():
#         try:
#             with zipfile.ZipFile(legacy_archive) as z:
#                 if z.testzip() is None:
#                     downloaded = legacy_archive
#         except (OSError, zipfile.BadZipFile, zlib.error, EOFError):
#             pass
#     if downloaded is None:
#         downloaded = run_kaggle_download()
#     else:
#         print(f'[download] reusing completed legacy archive: {downloaded}')
#     if downloaded.is_dir() and (downloaded.resolve().is_relative_to(data_dir.resolve())
#                                 or data_dir.resolve().is_relative_to(downloaded.resolve())):
#         raw = find_raw_data(downloaded, require_images=True)
#         if raw is not None:
#             return raw  # reuse overlapping complete data without moving or modifying the provider cache
#         raise ValueError('KaggleHub cache overlaps DATA_DIR. Choose a separate DATA_DIR for extraction.')
#     with tempfile.TemporaryDirectory(prefix='.extract-data-', dir=data_dir.parent) as tmp:
#         extracted = Path(tmp) / 'extracted'
#         if downloaded.is_dir():
#             # KaggleHub normally already extracted the competition archive. Work on a copy:
#             # its cache (or Kaggle's read-only input mount) must not be moved or unpacked in place.
#             shutil.copytree(downloaded, extracted, copy_function=shutil.copyfile)
#             for directory, _, _ in os.walk(extracted):
#                 Path(directory).chmod(0o700)  # only the working copy must allow nested ZIP extraction
#         elif zipfile.is_zipfile(downloaded):
#             extract_zip_checked(downloaded, extracted)
#         else:
#             raise ValueError(f'Expected a competition folder or ZIP from KaggleHub, received {downloaded}')
#         unpacked = set()
#         for _ in range(4):
#             raw = find_raw_data(extracted, require_images=True)
#             if raw is not None:
#                 publish_data_directory(raw, data_dir)
#                 print(f'[data] ready: {data_dir}')
#                 return data_dir
#             nested = [p for p in extracted.rglob('*.zip') if p not in unpacked]
#             if not nested:
#                 break
#             for path in nested:
#                 target = path.parent
#                 if path.stem in ('train_images', 'test_images'):
#                     with zipfile.ZipFile(path) as z:
#                         if not any(name.startswith(path.stem + '/') for name in z.namelist()):
#                             target = target / path.stem
#                 extract_zip_checked(path, target)
#                 unpacked.add(path)
#     raise ValueError(f'Downloaded files at {downloaded} lack complete train_images/, test_images/ and '
#                      'the three CSVs. Inspect that folder or set DATA_DIR to complete extracted data. '
#                      'If the KaggleHub cache is incomplete, refresh it with '
#                      'kagglehub.competition_download(..., force_download=True).')
#
#

def prepare_uploaded_data(uploaded, data_dir):
    """Read a user-supplied folder/ZIP; unpack only into local staging, without network access."""
    uploaded, data_dir = Path(uploaded), Path(data_dir)
    if not uploaded.exists():
        raise FileNotFoundError(
            f'No complete data or valid cache. Upload the competition ZIP or extracted files to '
            f'{uploaded}. On Colab the default is MyDrive/DL_Project1/data/. '
            'Kaggle downloads and login are disabled.')
    if (uploaded.resolve().is_relative_to(data_dir.resolve())
            or data_dir.resolve().is_relative_to(uploaded.resolve())):
        raise ValueError('Uploaded source and local extraction directory must be separate.')
    data_dir.parent.mkdir(parents=True, exist_ok=True)
    print(f'[data] reading manually uploaded files: {uploaded}')
    with tempfile.TemporaryDirectory(prefix='.extract-data-', dir=data_dir.parent) as tmp:
        extracted = Path(tmp) / 'extracted'
        if uploaded.is_dir():
            shutil.copytree(uploaded, extracted, copy_function=shutil.copyfile)
            for directory, _, _ in os.walk(extracted):
                Path(directory).chmod(0o700)
        elif zipfile.is_zipfile(uploaded):
            extract_zip_checked(uploaded, extracted)
        else:
            raise ValueError(f'Expected an uploaded competition folder or ZIP, received {uploaded}')
        unpacked = set()
        for _ in range(4):
            raw = find_raw_data(extracted, require_images=True)
            if raw is not None:
                publish_data_directory(raw, data_dir)
                print(f'[data] ready: {data_dir}')
                return data_dir
            nested = [p for p in extracted.rglob('*.zip') if p not in unpacked]
            if not nested:
                break
            for path in nested:
                target = path.parent
                if path.stem in ('train_images', 'test_images'):
                    with zipfile.ZipFile(path) as z:
                        if not any(name.startswith(path.stem + '/') for name in z.namelist()):
                            target = target / path.stem
                extract_zip_checked(path, target)
                unpacked.add(path)
    raise ValueError(
        f'Uploaded files at {uploaded} lack complete train_images/, test_images/, '
        'train_labels.csv, test.csv and sample_submission.csv. Upload all competition files; '
        'ZIP and extracted-folder layouts are supported.')


def prepare_data(paths, auto_download=False, save_to_drive=True):
    """Valid local cache -> compatible Drive cache -> manually uploaded folder/ZIP."""
    if auto_download:
        raise ValueError("Kaggle download/login is disabled. Keep AUTO_DOWNLOAD=False and upload data manually.")
    cache = paths.cache_dir
    persist = paths.env == 'colab' and save_to_drive
    if persist and not drive_mounted():
        raise RuntimeError('Mount Google Drive before data setup to save cache.zip, or explicitly set '
                           'SAVE_CACHE_TO_DRIVE=False for temporary storage only.')
    source = find_raw_data(paths.data_dir) or paths.data_dir
    meta = None
    if (cache / 'meta.json').is_file():
        try:
            candidate = verify_cache(cache)
            if _cache_matches_source(candidate, source):
                meta = candidate
                print(f'[cache] reusing verified local cache: {cache}')
            else:
                print('[cache] source CSVs changed; rebuilding from the requested data')
        except (OSError, ValueError, KeyError, RuntimeError, EOFError) as exc:
            print(f'[cache] local cache is incomplete or invalid ({type(exc).__name__})')
    if meta is None and paths.env == 'colab' and drive_mounted():
        try:
            fetch_drive_cache(drive_dir(), cache, source)
            meta = verify_cache(cache)
        except FileNotFoundError:
            pass
    if meta is None:
        if paths.data_dir.resolve() == cache.resolve():
            raise ValueError('Use separate DATA_DIR and CACHE_DIR when building from uploaded data')
        raw = find_raw_data(paths.data_dir, require_images=True)
        if raw is None:
            raw = prepare_uploaded_data(paths.data_dir, cache.parent / 'data_extracted')
        paths.data_dir = raw
        cache.parent.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(prefix='.build-cache-', dir=cache.parent) as tmp:
            staged = Path(tmp) / 'cache'
            build_cache(raw, staged)
            meta = verify_cache(staged)
            publish_data_directory(staged, cache)
    else:
        paths.data_dir = source
    if persist:
        save_drive_cache(cache, drive_dir())
    return cache, meta


In [ ]:
CACHE, META = prepare_data(PATHS, auto_download=AUTO_DOWNLOAD, save_to_drive=SAVE_CACHE_TO_DRIVE)
# Keep later training/evaluation cells on the resolved source folder (including wrapper directories).
DATA_DIR = str(PATHS.data_dir)
CACHE_DIR = str(CACHE)
print(f"[cache] {CACHE}: format {META['format']} | {META['n_train']:,} train / {META['n_test']:,} test images "
      f"{tuple(META['shape'])} | fingerprint {META['fingerprint'][:12]} | sha256 verified")
counts = np.bincount(np.load(CACHE / "train_labels.npy"), minlength=NUM_CLASSES)
print("Training images per class: " + ", ".join(f"{c} {n:,}" for c, n in zip(CLASSES, counts)))

In [ ]:
# Verbatim copies of E_pyramid_t106_shakedrop.ipynb helpers (split, normalization statistics, crop padding,
# inference, checkpoint identity, submission check). Do not edit: tests/test_q05_shift_tta.py compares every
# definition below with that notebook. model_fingerprint, validate_checkpoint and load_model use section 4's
# model code (MODEL, build_model, ...).
def _per_class(n, what):
    if not isinstance(n, (int, np.integer)) or isinstance(n, (bool, np.bool_)) or n < 0:
        raise ValueError(f"{what} must be a nonnegative integer, got {n!r}")
    if n % NUM_CLASSES:
        raise ValueError(f"{what} must be a multiple of {NUM_CLASSES} for a class-balanced split, got {n}")
    return n // NUM_CLASSES


def stratified_split(labels, val_size, seed=0):
    """Hold out `val_size` images, the same number from every class. Returns sorted (train_idx, val_idx)."""
    labels = np.asarray(labels)
    if labels.ndim != 1 or not np.issubdtype(labels.dtype, np.integer) or not len(labels):
        raise ValueError("labels must be a nonempty one-dimensional integer array")
    if np.any((labels < 0) | (labels >= NUM_CLASSES)):
        raise ValueError("labels must be in 0..9")
    k = _per_class(val_size, "val_size")
    if val_size >= len(labels):
        raise ValueError("Validation size must leave at least one training image")
    if np.any(np.bincount(labels, minlength=NUM_CLASSES) < k):
        raise ValueError("Not enough examples per class for the requested validation split")
    all_idx = np.arange(len(labels))
    if val_size == 0:
        return all_idx, np.array([], dtype=np.int64)
    k = _per_class(val_size, "val_size")
    rng = np.random.default_rng(seed)
    val = np.concatenate([rng.permutation(np.flatnonzero(labels == c))[:k] for c in range(NUM_CLASSES)])
    val = np.sort(val)
    return np.setdiff1d(all_idx, val), val


def channel_stats(images):
    """Per-channel mean/std in [0, 1] units over uint8 NHWC images (computed in chunks)."""
    s = np.zeros(3)
    ss = np.zeros(3)
    n = 0
    for i in range(0, len(images), 5000):
        chunk = np.asarray(images[i:i + 5000], dtype=np.float64) / 255.0
        s += chunk.sum(axis=(0, 1, 2))
        ss += (chunk ** 2).sum(axis=(0, 1, 2))
        n += chunk.shape[0] * chunk.shape[1] * chunk.shape[2]
    mean = s / n
    std = np.sqrt(ss / n - mean ** 2)
    return mean.tolist(), std.tolist()


def to_float_nchw(x_u8, mean, std, channels_last=False):
    """uint8 NHWC on device -> normalized float NCHW (channels_last strides are free from NHWC)."""
    x = x_u8.permute(0, 3, 1, 2).float().div_(255.0)
    x = (x - mean) / std
    return x.contiguous(memory_format=torch.channels_last if channels_last else torch.contiguous_format)


def crop_pad(x_u8, pad=4, padding_mode="constant"):
    """Pad uint8 NHWC images before random cropping; reflect excludes the edge pixel (same code as scheme C/T31)."""
    if padding_mode not in ("constant", "reflect"):
        raise ValueError("crop_padding_mode must be constant or reflect")
    if padding_mode == "constant":
        return F.pad(x_u8, (0, 0, pad, pad, pad, pad))  # original black-padding path
    h, w = x_u8.shape[1:3]
    if not isinstance(pad, int) or not 0 <= pad < min(h, w):
        raise ValueError("Reflection padding must be nonnegative and smaller than the image")
    # Integer indexing preserves uint8 on both CPU and CUDA, with no random draws.
    rows = torch.arange(-pad, h + pad, device=x_u8.device)
    cols = torch.arange(-pad, w + pad, device=x_u8.device)
    rows = torch.where(rows < 0, -rows, torch.where(rows >= h, 2 * h - 2 - rows, rows))
    cols = torch.where(cols < 0, -cols, torch.where(cols >= w, 2 * w - 2 - cols, cols))
    return x_u8.index_select(1, rows).index_select(2, cols)


def replace_file(src, dst):
    """Publish a completed file by rename; never overwrite a good checkpoint by partial copy."""
    try:
        os.replace(src, dst)
    except OSError as exc:
        raise OSError(f"Could not atomically publish {dst}; the previous file is preserved. "
                      "Use a filesystem that supports same-directory rename.") from exc


def write_bytes_atomic(data, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_bytes(data)
    replace_file(tmp, path)


def load_checkpoint(path):
    state = torch.load(path, map_location="cpu", weights_only=False)
    if not isinstance(state, dict) or "args" not in state:
        return state
    original = state["args"].get("run_name")
    canonical = canonical_run_id(original)
    if original != canonical:
        state.setdefault("original_identity", {"run_id": original,
                         "model_version": state["args"].get("model_version")})
        state["args"] = dict(state["args"], run_name=canonical,
                             model_version=int(re.search(r"-ver-(\d+)-", canonical).group(1)),
                             task_id=run_task_id(canonical))
    return state


@torch.inference_mode()
def predict_logits(model, images_u8, mean, std, device, batch_size=1000, flip=False, channels_last=False):
    """Logits for uint8 NHWC images (numpy or tensor); returns a CPU float tensor.

    With flip=True, returns the logits of horizontally flipped images instead.
    """
    model.eval()
    mean_t = torch.tensor(mean, device=device).view(1, 3, 1, 1)
    std_t = torch.tensor(std, device=device).view(1, 3, 1, 1)
    out = []
    for i in range(0, len(images_u8), batch_size):
        xb = torch.as_tensor(np.asarray(images_u8[i:i + batch_size])).to(device)
        x = to_float_nchw(xb, mean_t, std_t, channels_last)
        if flip:
            x = x.flip(3)
        out.append(model(x).float().cpu())
    return torch.cat(out)


def predict_probs(model, images_u8, mean, std, device, batch_size=1000, tta=True, channels_last=False):
    """Softmax probabilities; with tta=True, averages the original and horizontally flipped view."""
    kw = dict(batch_size=batch_size, channels_last=channels_last)
    probs = F.softmax(predict_logits(model, images_u8, mean, std, device, **kw), 1)
    if tta:
        probs = 0.5 * (probs + F.softmax(predict_logits(model, images_u8, mean, std, device, flip=True, **kw), 1))
    return probs


CHECKPOINT_FORMAT = 3


def write_json_atomic(value, path):
    write_bytes_atomic((json.dumps(value, indent=2) + "\n").encode(), path)


def model_fingerprint(model):
    options = getattr(model, "_project_architecture", ARCH_DEFAULTS)
    definition = getattr(model, "_project_description", MODEL_DESCRIPTION) + "\n" + repr(model)
    # Preserve the exact legacy baseline hash. New variants also bind options and implementation version.
    if options != ARCH_DEFAULTS:
        definition += "\n" + json.dumps(options, sort_keys=True)
    return hashlib.sha256(definition.encode()).hexdigest()


def validate_checkpoint(ckpt, fingerprint, expected_model_hash=None, run_id=None):
    if ckpt.get("format") != CHECKPOINT_FORMAT or "cache_fingerprint" not in ckpt:
        raise ValueError("Legacy checkpoint lacks verified dataset identity; start a fresh run with this notebook")
    if ckpt["cache_fingerprint"] != fingerprint:
        raise ValueError("Checkpoint dataset fingerprint differs from the current cache; refusing resume/evaluation")
    if ckpt["args"]["model"] != MODEL:
        raise ValueError("Checkpoint belongs to another model")
    saved_options = saved_architecture(ckpt["args"])
    if "architecture" in ckpt and ckpt["architecture"] != architecture_details(saved_options):
        raise ValueError("Checkpoint resolved architecture differs from its saved options")
    if expected_model_hash is not None and ckpt.get("model_fingerprint") != expected_model_hash:
        raise ValueError("Checkpoint architecture differs from this notebook")
    if run_id is not None and canonical_run_id(ckpt["args"]["run_name"]) != canonical_run_id(run_id):
        raise ValueError("Checkpoint belongs to another run ID")


def load_model(ckpt, weights, device):
    options = saved_architecture(ckpt["args"])
    model, n_params = build_model(architecture=options)
    if "architecture" in ckpt and ckpt["architecture"] != architecture_details(options):
        raise ValueError("Checkpoint resolved architecture differs from its saved options")
    if "n_params" in ckpt and ckpt["n_params"] != n_params:
        raise ValueError("Checkpoint parameter count differs from its saved architecture")
    if ckpt.get("model_fingerprint") != model_fingerprint(model) or ckpt["args"]["model"] != MODEL:
        raise ValueError("Checkpoint architecture differs from this notebook")
    model.load_state_dict(ckpt["ema"]["module"] if weights == "ema" else ckpt["model"])
    return model.to(device).eval(), n_params


def validate_submission(path, sample_path):
    sub, sample = pd.read_csv(path), pd.read_csv(sample_path)
    if list(sub.columns) != ["id", "label"] or len(sub) != len(sample):
        raise ValueError("Submission columns or row count differ from the sample")
    if not sub["id"].is_unique or not np.array_equal(sub["id"], sample["id"]):
        raise ValueError("Submission IDs are duplicated or differ from sample order")
    if not pd.api.types.is_integer_dtype(sub["label"]) or not sub["label"].between(0,9).all():
        raise ValueError("Submission labels must be integers in 0..9")
    return len(sub)


def classification_metrics(probs, labels):
    labels = np.asarray(labels)
    probs = np.asarray(probs)
    pred = probs.argmax(1)
    cm = np.bincount(labels * NUM_CLASSES + pred, minlength=NUM_CLASSES**2).reshape(NUM_CLASSES, NUM_CLASSES)
    support = cm.sum(1)
    recall = np.divide(cm.diagonal(), support, out=np.zeros(NUM_CLASSES, dtype=float), where=support > 0)
    return {"accuracy_pct": float((pred == labels).mean()*100), "correct": int((pred == labels).sum()),
            "n": len(labels), "cross_entropy": float(-np.log(probs[np.arange(len(labels)), labels].clip(1e-12)).mean()),
            "per_class_recall": {name: float(value) if count else None
                                 for name, value, count in zip(CLASSES, recall, support)},
            "confusion_matrix": cm.tolist()}

In [ ]:
# ---- the training notebooks' exact 45k/5k split, their training-split normalization, and the test images
COMPETITION_CACHE_FINGERPRINT = "42749a486a13184f0f96dab13293950f0120c357ddf1e6b1ef6a3bd3a9ce5fde"
# SHA-256 of the sorted int64 validation indices and their labels (the values Q08_shiftval.ipynb pins). Both equal
# every validation export saved by the A/B/C/E/F/G runs (result/evaluations/validation/<RUN_ID>/*-validation.npz).
EXPECTED_VAL_INDICES_SHA256 = "7957d060fcc34aefa5a43c70e31433927db55526494f67a9f7da553059be6767"
EXPECTED_VAL_LABELS_SHA256 = "aba30157995bda52486927048b705427db4a51e03e0bcb4844d462bb15f20846"


def int64_sha256(values):
    """SHA-256 of little-endian int64 bytes (indices and labels)."""
    return hashlib.sha256(np.ascontiguousarray(values, dtype="<i8").tobytes()).hexdigest()


if Q05_COMPETITION_CHECKS:
    if META["fingerprint"] != COMPETITION_CACHE_FINGERPRINT:
        raise RuntimeError(f"Cache fingerprint {META['fingerprint'][:12]} is not the competition dataset "
                           f"{COMPETITION_CACHE_FINGERPRINT[:12]}; Q05 must use the training notebooks' data")
    if (VAL_SIZE, SPLIT_SEED) != (5000, 0):
        raise ValueError("Q05 uses the training notebooks' split: keep VAL_SIZE=5000 and SPLIT_SEED=0")
_train_images, _train_labels, _train_ids = load_train(DATA_DIR, CACHE_DIR, mmap=True)
TRAIN_IDX, VAL_IDX = stratified_split(_train_labels, VAL_SIZE, SPLIT_SEED)
if not len(VAL_IDX):
    raise ValueError("Q05 needs a validation split (VAL_SIZE > 0)")
VAL_LABELS_ALL = np.ascontiguousarray(_train_labels[VAL_IDX], dtype=np.int64)
if Q05_COMPETITION_CHECKS:
    if int64_sha256(VAL_IDX) != EXPECTED_VAL_INDICES_SHA256 or int64_sha256(VAL_LABELS_ALL) != EXPECTED_VAL_LABELS_SHA256:
        raise RuntimeError("Validation indices or labels differ from the training notebooks' 5k split")
# train() computes the normalization with channel_stats on the training split; every checkpoint's own saved statistics
# are used for evaluation (as in training), and each is compared with this recomputation.
TRAIN_NORM = channel_stats(_train_images[TRAIN_IDX])
if type(Q05_SMOKE_N) is not int or Q05_SMOKE_N < 0:
    raise ValueError("Q05_SMOKE_N must be a nonnegative integer")
Q05_N = min(Q05_SMOKE_N, len(VAL_IDX)) if Q05_SMOKE_N else len(VAL_IDX)
Q05_SMOKE = Q05_N != len(VAL_IDX)
VAL_X = np.ascontiguousarray(_train_images[VAL_IDX[:Q05_N]])  # raw uint8 NHWC; normalized per checkpoint
VAL_Y = VAL_LABELS_ALL[:Q05_N]
del _train_images, _train_labels, _train_ids
# Test images are only forwarded, one image per view, for the optional CSVs; no statistic is computed from them.
TEST_X, TEST_IDS = load_test(DATA_DIR, CACHE_DIR) if Q05_WRITE_TEST_CSVS and not Q05_SMOKE else (None, None)
print(f"[split] {len(TRAIN_IDX):,} train / {len(VAL_IDX):,} validation (seed {SPLIT_SEED}); "
      f"indices sha256 {int64_sha256(VAL_IDX)[:12]} | labels sha256 {int64_sha256(VAL_LABELS_ALL)[:12]} | "
      f"cache {META['fingerprint'][:12]}" + (" | competition split verified" if Q05_COMPETITION_CHECKS else ""))
print("Training-split normalization: mean " + ", ".join(f"{v:.6f}" for v in TRAIN_NORM[0])
      + " | std " + ", ".join(f"{v:.6f}" for v in TRAIN_NORM[1]))
print("Validation images per class: " + ", ".join(
    f"{c} {n}" for c, n in zip(CLASSES, np.bincount(VAL_LABELS_ALL, minlength=NUM_CLASSES))))
if TEST_X is not None:
    print(f"[test] {len(TEST_X):,} test images for the optional shift5x2 / grid9x2 CSVs")
if Q05_SMOKE:
    print(f"Warning: Q05_SMOKE_N={Q05_SMOKE_N}: using the first {Q05_N} validation images; results are a smoke "
          "test, not comparable; reproduction is not checked and no test CSV is written")

## 4. Model: verbatim E code of `E_pyramid_t106_shakedrop.ipynb`

The next two cells are verbatim copies of that notebook's section 4. They rebuild basic PyramidNet E (E1/T18 and every
DropPath run, T106 included, with their original model fingerprints), ShakeDrop and bottleneck PyramidNets (T122) from
the options saved with each checkpoint; the baseline switch values in section 1 only feed the cell's own baseline
build. In evaluation mode DropPath is the identity, ShakeDrop multiplies each residual branch by its expectation
1 − p_l, and MixStyle (E9/T89) is not applied. T118 was trained with `E_trivialaug_ibn.ipynb`, whose four IBN options
are all off there: its network is this basic E with DropPath 0.1, so it is rebuilt with this code after its stored
fingerprint has matched the value pinned in section 5. Head256/multistage heads and active IBN options are not
rebuilt.

In [ ]:
MODEL = "E"
MODEL_DESCRIPTION = "PyramidNet-110 basic, alpha=98 (widths 16->114)"  # the default (E1/T18, T106) architecture
EXPECTED_PARAMS = 4_963_991  # default basic-110, alpha 98; other BLOCK / DEPTH / ALPHA values: expected_param_count()


class DropPath(nn.Module):
    """Stochastic depth (Huang et al., 2016) for one residual branch, per sample.

    In training, each image's branch is dropped with probability p and survivors are scaled by 1 / (1 - p);
    in evaluation it is the identity. The mask comes from the global torch RNG of the input's device.
    """

    def __init__(self, p=0.0):
        super().__init__()
        self.p = float(p)

    def forward(self, x):
        if not self.training or self.p == 0.0:
            return x
        keep = 1.0 - self.p
        mask = torch.rand(x.shape[0], 1, 1, 1, device=x.device) < keep
        return x * mask.to(x.dtype) / keep

    def extra_repr(self):
        return f"p={self.p:.4f}"


class ShakeDrop(nn.Module):
    """ShakeDrop (Yamada, Iwamura, Akiba & Kise, 2019, arXiv 1802.02375) on one residual branch output F(x).

    p is the probability that this block is perturbed. Training: a gate b ~ Bernoulli(1 - p) per image, then
    alpha ~ U[-1, 1] and beta ~ U[0, 1] per element of F(x) (the paper's pixel-level rule). The forward pass
    multiplies F(x) by b + alpha - b*alpha, the backward pass multiplies the gradient by b + beta - b*beta: a kept
    branch (b = 1) is untouched, a perturbed one is scaled by alpha forward and by beta backward. Evaluation multiplies
    F(x) by E[b + alpha - b*alpha] = 1 - p. Like DropPath, the draws come from the default RNG of the input's device
    (gate, then alpha, then beta), inside the (compiled) training graph; no custom autograd.Function is needed:
    y = beta_coef * F + (alpha_coef - beta_coef) * F.detach() has the forward value alpha_coef * F and the gradient
    beta_coef. Coefficients and the product are computed in float32 and cast back to F's dtype (bf16 under autocast);
    channels_last is preserved. No parameters or buffers.
    """

    def __init__(self, p=0.0):
        super().__init__()
        self.p = float(p)

    def coefficients(self, x):
        """(forward, backward) float32 coefficients for a training batch shaped like x (one gate per image).

        b + alpha - b*alpha is computed as b + (1 - b) * alpha, which is exactly 1 for a kept branch (b = 1) and exactly
        alpha for a perturbed one (b = 0); likewise for beta.
        """
        keep = 1.0 - self.p
        gate = (torch.rand(x.shape[0], 1, 1, 1, device=x.device) < keep).to(torch.float32)
        alpha = torch.rand_like(x, dtype=torch.float32) * 2.0 - 1.0  # U[-1, 1]
        beta = torch.rand_like(x, dtype=torch.float32)  # U[0, 1]
        perturbed = 1.0 - gate
        return gate + perturbed * alpha, gate + perturbed * beta

    def forward(self, x):
        if self.p == 0.0:
            return x
        if not self.training:
            return x * (1.0 - self.p)  # E[b + alpha - b*alpha] = keep probability
        forward, backward = self.coefficients(x)
        x32 = x.float()
        return (x32 * backward + x32.detach() * (forward - backward)).to(x.dtype)

    def extra_repr(self):
        return f"p={self.p:.4f}"


def sam_replay_needed(model):
    """True if the training forward draws ShakeDrop coefficients: SAM's second pass then replays the first pass's
    draws by restoring the device RNG (SharpnessAwareMinimization, section 5). DropPath masks stay independent."""
    return any(isinstance(m, ShakeDrop) and m.p > 0 for m in model.modules())


MIXSTYLE_AFTER_STAGES = (1, 2)  # training-only MixStyle (MIXSTYLE, section 1) mixes the outputs of stages 1 and 2


def mix_feature_styles(x, apply, lam, perm, eps):
    """MixStyle (Zhou et al., ICLR 2021), "random" variant, on one training batch; parameter-free.

    Each image's per-channel mean mu and standard deviation sigma over the spatial positions (unbiased variance + eps,
    computed in float32 and detached, as in the reference code) are replaced by lam * stat + (1 - lam) * stat of its
    partner image perm[i]; the result is cast back to x's dtype. `apply` (0-dim bool tensor) selects the mixed batch
    or exactly x; lam has shape (B, 1, 1, 1). The draws come from the trainer's MixStyleSampler (section 5).
    """
    x32 = x.float()
    mu = x32.mean(dim=(2, 3), keepdim=True).detach()
    sigma = (x32.var(dim=(2, 3), keepdim=True) + eps).sqrt().detach()
    mixed_mu = lam * mu + (1 - lam) * mu[perm]
    mixed_sigma = lam * sigma + (1 - lam) * sigma[perm]
    mixed = ((x32 - mu) / sigma * mixed_sigma + mixed_mu).to(x.dtype)
    return torch.where(apply, mixed, x)


# ACTIVATION (section 1): the nonlinearity inside every residual branch and before global pooling. A plain function,
# not a module: no parameter, buffer or repr line, so ReLU models keep their state_dict keys, repr, description and
# fingerprints. SiLU (x * sigmoid(x), Elfwing et al., 2018) has no parameters either.
ACTIVATIONS = {"relu": F.relu, "silu": F.silu}


class PyramidBlock(nn.Module):
    """BN-conv-BN-ReLU-conv-BN (Han et al., 2017), added to a parameter-free shortcut; no ReLU after the addition.

    The ShakeDrop module is registered only when its rate is positive, so the default block (and the T106 model
    fingerprint, which includes repr(model)) is unchanged. `activation` (a key of ACTIVATIONS) replaces the ReLU.
    """

    def __init__(self, in_ch, out_ch, stride=1, drop_path=0.0, shakedrop=0.0, activation="relu"):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_ch)
        self.drop_path = DropPath(drop_path)
        self.shakedrop = ShakeDrop(shakedrop) if shakedrop else None
        self.act = ACTIVATIONS[activation]  # plain function attribute: not a module, not in repr or state_dict
        # Shortcut: identity, 2x2 average pool where the block downsamples, zero padding for the extra channels.
        self.pool = nn.AvgPool2d(2, 2, ceil_mode=True) if stride != 1 else nn.Identity()
        self.extra_ch = out_ch - in_ch

    def forward(self, x):
        out = self.conv1(self.bn1(x))
        out = self.conv2(self.act(self.bn2(out)))
        out = self.drop_path(self.bn3(out))
        if self.shakedrop is not None:  # F(x) before the shortcut addition
            out = self.shakedrop(out)
        shortcut = self.pool(x)
        if self.extra_ch:
            shortcut = F.pad(shortcut, (0, 0, 0, 0, 0, self.extra_ch))
        return out + shortcut


class PyramidBottleneck(nn.Module):
    """Bottleneck PyramidNet block (Han et al., 2017; dyhan0920/PyramidNet-PyTorch): BN-conv1x1-BN-ReLU-conv3x3
    (stride)-BN-ReLU-conv1x1(x4)-BN, added to the same parameter-free shortcut (2x2 average pool where the block
    downsamples, zero-padded channels); no ReLU after the addition. `planes` is the pyramid width; the block outputs
    4 * planes channels. `activation` (a key of ACTIVATIONS) replaces both ReLUs."""

    expansion = 4

    def __init__(self, in_ch, planes, stride=1, drop_path=0.0, shakedrop=0.0, activation="relu"):
        super().__init__()
        out_ch = planes * self.expansion
        self.bn1 = nn.BatchNorm2d(in_ch)
        self.conv1 = nn.Conv2d(in_ch, planes, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, stride, 1, bias=False)
        self.bn3 = nn.BatchNorm2d(planes)
        self.conv3 = nn.Conv2d(planes, out_ch, 1, bias=False)
        self.bn4 = nn.BatchNorm2d(out_ch)
        self.drop_path = DropPath(drop_path)
        self.shakedrop = ShakeDrop(shakedrop) if shakedrop else None
        self.act = ACTIVATIONS[activation]  # plain function attribute: not a module, not in repr or state_dict
        self.pool = nn.AvgPool2d(2, 2, ceil_mode=True) if stride != 1 else nn.Identity()
        self.extra_ch = out_ch - in_ch

    def forward(self, x):
        out = self.conv1(self.bn1(x))
        out = self.conv2(self.act(self.bn2(out)))
        out = self.conv3(self.act(self.bn3(out)))
        out = self.drop_path(self.bn4(out))
        if self.shakedrop is not None:
            out = self.shakedrop(out)
        shortcut = self.pool(x)
        if self.extra_ch:
            shortcut = F.pad(shortcut, (0, 0, 0, 0, 0, self.extra_ch))
        return out + shortcut


# BLOCK -> (layers per block, so depth = 3 * layers * n + 2, and the block's output channels per pyramid width)
PYRAMID_BLOCKS = {"basic": (2, 1), "bottleneck": (3, PyramidBottleneck.expansion)}


def blocks_per_stage(depth, block):
    """n blocks per stage: (depth - 2) / 6 for basic blocks, (depth - 2) / 9 for bottlenecks."""
    if block not in PYRAMID_BLOCKS:
        raise ValueError(f"BLOCK = {block!r}; choose one of {tuple(PYRAMID_BLOCKS)}")
    per_stage = 3 * PYRAMID_BLOCKS[block][0]
    if type(depth) is not int or depth < per_stage + 2 or (depth - 2) % per_stage:
        raise ValueError(f"DEPTH = {depth!r} is invalid for {block} blocks: use {per_stage}n + 2 with n >= 1 "
                         f"(e.g. {'110' if block == 'basic' else '164, 200 or 272'})")
    return (depth - 2) // per_stage


class PyramidNet(nn.Module):
    """3x3 stem (16 channels, BN), 3 stages of n pyramidal blocks, BN-ReLU, global average pool, linear.

    n = (depth-2)/6 basic or (depth-2)/9 bottleneck blocks per stage. The width grows by alpha / N after each of the
    N = 3n blocks. As in the reference implementation, the width is accumulated as a float and each block uses
    int(round(width)) channels (bottlenecks output 4x that). Block l of N drops its residual branch with probability
    drop_path * l / N (linear decay rule of Huang et al., 2016), or, with ShakeDrop, is perturbed with probability
    shakedrop * l / N (Yamada et al.'s linear rule with p_L = 1 - shakedrop); the two are exclusive. A training forward
    may receive MixStyle draws for the points after stages MIXSTYLE_AFTER_STAGES; this adds no module, parameter or
    buffer. `activation` (a key of ACTIVATIONS, default "relu") is the nonlinearity inside every block and of the final
    BN-activation before pooling; nowhere else.
    """

    mixstyle_layers = len(MIXSTYLE_AFTER_STAGES)  # number of MixStyle draws a training forward accepts

    def __init__(self, depth=110, alpha=98, drop_path=0.0, num_classes=10, block="basic", shakedrop=0.0,
                 activation="relu"):
        super().__init__()
        if drop_path and shakedrop:
            raise ValueError("DropPath and ShakeDrop act on the same residual branch; enable only one of them")
        if not isinstance(activation, str) or activation not in ACTIVATIONS:
            raise ValueError(f"activation = {activation!r}; choose one of {tuple(ACTIVATIONS)}")
        n = blocks_per_stage(depth, block)
        unit = PyramidBlock if block == "basic" else PyramidBottleneck
        expansion = PYRAMID_BLOCKS[block][1]
        total = 3 * n
        add_rate = alpha / total
        self.stem = nn.Sequential(nn.Conv2d(3, 16, 3, 1, 1, bias=False), nn.BatchNorm2d(16))
        stages, in_ch, width, index = [], 16, 16.0, 0
        for i in range(3):
            layers = []
            for j in range(n):
                index += 1
                width += add_rate
                planes = int(round(width))
                stride = 2 if (i > 0 and j == 0) else 1
                layers.append(unit(in_ch, planes, stride, drop_path * index / total, shakedrop * index / total,
                                   activation))
                in_ch = planes * expansion
            stages.append(nn.Sequential(*layers))
        self.stages = nn.Sequential(*stages)
        self.bn = nn.BatchNorm2d(in_ch)
        self.fc = nn.Linear(in_ch, num_classes)
        self.act = ACTIVATIONS[activation]  # final BN-activation; a plain function, as in the blocks

    def forward(self, x, mixstyle=None):
        """`mixstyle` = (apply, lam, perm, eps) from MixStyleSampler.draw, or None. Without draws, and always in
        evaluation mode, this is exactly the plain PyramidNet forward."""
        x = self.stem(x)
        if mixstyle is None or not self.training:
            x = self.stages(x)
        else:
            apply, lam, perm, eps = mixstyle
            layer = 0
            for index, stage in enumerate(self.stages, 1):
                x = stage(x)
                if index in MIXSTYLE_AFTER_STAGES:
                    x = mix_feature_styles(x, apply[layer], lam[layer], perm[layer], eps)
                    layer += 1
        x = self.act(self.bn(x))
        return self.fc(torch.flatten(F.adaptive_avg_pool2d(x, 1), 1))


def pyramid_widths(depth=110, alpha=98, block="basic"):
    """Pyramid width of every block and output channels of each stage, by PyramidNet's int(round(width)) rule."""
    n = blocks_per_stage(depth, block)
    rate = alpha / (3 * n)
    width, planes = 16.0, []
    for _ in range(3 * n):
        width += rate
        planes.append(int(round(width)))
    expansion = PYRAMID_BLOCKS[block][1]
    return planes, [planes[(i + 1) * n - 1] * expansion for i in range(3)]


# Always-present options (unchanged since E1) and optional ones that are omitted from the saved options when at their
# default, so the E1/T18 and T106 options, descriptions and model fingerprints are reproduced exactly.
ARCH_DEFAULTS = {"implementation_version": 1, "drop_path": 0.0}
ARCH_OPTION_DEFAULTS = {"shakedrop": 0.0, "block": "basic", "depth": 110, "alpha": 98, "activation": "relu"}


def arch_option(options, key):
    return options.get(key, ARCH_OPTION_DEFAULTS[key])


def architecture_from_config():
    return {"drop_path": DROP_PATH, "shakedrop": SHAKEDROP, "block": BLOCK, "depth": DEPTH, "alpha": ALPHA,
            "activation": ACTIVATION}


def validate_architecture(options):
    """Check the resolved options in place (SHAKEDROP becomes a float, an integral float ALPHA an int)."""
    rate = options["drop_path"]
    if isinstance(rate, bool) or not isinstance(rate, (int, float)) or not 0 <= rate < 1:
        raise ValueError(f"DROP_PATH = {rate!r}; choose a rate in [0, 1) (0 = no stochastic depth)")
    shake = options["shakedrop"]
    if isinstance(shake, bool) or not isinstance(shake, (int, float)) or not 0 <= shake < 1:
        raise ValueError(f"SHAKEDROP = {shake!r}; choose a rate in [0, 1) (0 = off; 0.5 = the paper's p_L = 0.5)")
    options["shakedrop"] = float(shake)
    if rate > 0 and shake > 0:
        raise ValueError(f"DROP_PATH = {rate!r} and SHAKEDROP = {shake!r}: both act on the same residual branch; "
                         "set one of them to 0")
    blocks_per_stage(options["depth"], options["block"])  # checks BLOCK and DEPTH
    alpha = options["alpha"]
    if isinstance(alpha, bool) or not isinstance(alpha, (int, float)) or not math.isfinite(alpha) or alpha <= 0:
        raise ValueError(f"ALPHA = {alpha!r}; choose a finite positive widening (98 for basic-110)")
    if isinstance(alpha, float) and alpha.is_integer():
        options["alpha"] = int(alpha)  # 100.0 and 100 are the same architecture and fingerprint
    activation = options["activation"]
    if not isinstance(activation, str) or activation not in ACTIVATIONS:
        raise ValueError(f"ACTIVATION = {activation!r}; choose one of {tuple(ACTIVATIONS)}")
    count = expected_param_count(options)
    if count > PARAM_BUDGET:
        raise ValueError(f"{options['block']} PyramidNet-{options['depth']} with alpha {options['alpha']} has "
                         f"{count:,} parameters, over the {PARAM_BUDGET:,} budget")


def describe_architecture(options):
    # The rates are saved with the options and enter the architecture fingerprint: resuming with another rate is
    # rejected, and evaluation rebuilds the saved options whatever the current switches. For the default options the
    # first part is exactly MODEL_DESCRIPTION.
    block, depth, alpha = arch_option(options, "block"), arch_option(options, "depth"), arch_option(options, "alpha")
    planes, ends = pyramid_widths(depth, alpha, block)
    text = (f"PyramidNet-{depth} {block}, alpha={alpha} (widths 16->{planes[-1]}"
            + (f", block outputs {planes[0] * PYRAMID_BLOCKS[block][1]}->{ends[-1]}" if block != "basic" else "")
            + f"), stochastic depth DROP_PATH={float(options['drop_path'])!r}")
    shake = arch_option(options, "shakedrop")
    if shake:
        total = 3 * blocks_per_stage(depth, block)
        text += (f", ShakeDrop SHAKEDROP={float(shake)!r} (block l of L={total} perturbed with probability "
                 f"{float(shake)!r}*l/L, p_L={1 - shake:g}; gate per image, alpha/beta per element)")
    activation = arch_option(options, "activation")
    if activation != "relu":  # nothing is appended for ReLU, so earlier descriptions are reproduced exactly
        text += (f", ACTIVATION={activation!r} (in place of every ReLU: inside each residual branch and before global "
                 "pooling)")
    return text


def architecture_feature(options):
    if arch_option(options, "activation") != "relu":
        return arch_option(options, "activation")
    if arch_option(options, "block") != "basic":
        return "bottleneck"
    if arch_option(options, "shakedrop") > 0:
        return "shakedrop"
    return "stochdepth" if options["drop_path"] > 0 else None


def expected_param_count(options):
    """Exact trainable parameters from the widths (bias-free convolutions, BN gamma/beta, linear weight and bias).
    DropPath, ShakeDrop, MixStyle and the activation (ReLU or SiLU) add none."""
    block, depth, alpha = arch_option(options, "block"), arch_option(options, "depth"), arch_option(options, "alpha")
    planes, _ = pyramid_widths(depth, alpha, block)
    expansion = PYRAMID_BLOCKS[block][1]
    total, in_ch = 3 * 16 * 9 + 2 * 16, 16  # stem conv + BN
    for p in planes:
        out = p * expansion
        if block == "basic":
            total += 2 * in_ch + 9 * in_ch * p + 2 * p + 9 * p * p + 2 * p
        else:
            total += 2 * in_ch + in_ch * p + 2 * p + 9 * p * p + 2 * p + p * out + 2 * out
        in_ch = out
    return total + 2 * in_ch + in_ch * NUM_CLASSES + NUM_CLASSES  # final BN + linear


def architecture_topology(options):
    """Stem width, output channels of each stage (the width grows by alpha/N in every block) and blocks per stage."""
    block, depth, alpha = arch_option(options, "block"), arch_option(options, "depth"), arch_option(options, "alpha")
    n = blocks_per_stage(depth, block)
    return {"stem_channels": 16, "stage_channels": pyramid_widths(depth, alpha, block)[1], "blocks": [n, n, n]}


def make_model(num_classes=10, architecture=None):
    options = resolve_architecture(architecture)
    return PyramidNet(depth=arch_option(options, "depth"), alpha=arch_option(options, "alpha"),
                      drop_path=options["drop_path"], num_classes=num_classes, block=arch_option(options, "block"),
                      shakedrop=arch_option(options, "shakedrop"), activation=arch_option(options, "activation"))

In [ ]:
PARAM_BUDGET = 5_000_000


def init_weights(model):
    """Kaiming-normal (fan_out) convs and unit/zero BN, as in torchvision's ResNet. The ReLU gain is kept with
    ACTIVATION="silu" (unchanged init): a BatchNorm follows every convolution, so it only scales the BN input."""
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
        elif isinstance(m, nn.BatchNorm2d):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)
    return model


def count_params(model):
    """Number of trainable parameters (conv/linear weights and biases, BN gamma/beta)."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


ARCHITECTURE_IMPLEMENTATION_VERSION = 1  # Bump when the meaning of a switch/forward pass changes.


def resolve_architecture(options=None):
    """None reads UI switches; an explicit dict (including {}) uses saved/baseline options."""
    options = architecture_from_config() if options is None else options
    if not isinstance(options, dict):
        raise ValueError("Architecture options must be a dictionary")
    unknown = set(options) - (set(ARCH_DEFAULTS) | set(ARCH_OPTION_DEFAULTS))
    if unknown:
        raise ValueError(f"Unknown architecture options for model {MODEL}: {sorted(unknown)}")
    resolved = {**ARCH_DEFAULTS, **ARCH_OPTION_DEFAULTS, **options}
    version = resolved["implementation_version"]
    if type(version) is not int or version != ARCHITECTURE_IMPLEMENTATION_VERSION:
        raise ValueError("Unsupported architecture implementation_version; use the matching notebook code")
    validate_architecture(resolved)
    # Canonical omission: default-valued optional options (SHAKEDROP 0, basic blocks, depth 110, alpha 98, ReLU) are
    # dropped, so E1/T18 and T106 options, architecture records and fingerprints are reproduced exactly.
    return {k: v for k, v in resolved.items() if k not in ARCH_OPTION_DEFAULTS or v != ARCH_OPTION_DEFAULTS[k]}


def saved_architecture(args):
    """Missing legacy fields mean baseline; malformed saved options must never read UI globals."""
    options = args.get("architecture", ARCH_DEFAULTS)
    if not isinstance(options, dict):
        raise ValueError("Saved architecture options must be a dictionary")
    return resolve_architecture(options)


def architecture_details(options):
    """Resolved topology accompanies options in configs, checkpoints and evaluation reports."""
    details = resolve_architecture(options)
    if MODEL not in ("A", "B", "C"):  # E/F/G define architecture_topology next to their model (section 4)
        return dict(details, model=MODEL, **architecture_topology(details), num_classes=NUM_CLASSES)
    channels = WIDTH_PRESETS[details["width_preset"]] if MODEL == "B" else (64, 128, 256)
    return dict(details, model=MODEL, stem_channels=16 if MODEL == "B" else 64,
                stage_channels=list(channels), blocks=[3, 3, 3] if MODEL == "B" else [3, 5, 3],
                num_classes=NUM_CLASSES)


def build_model(num_classes=NUM_CLASSES, architecture=None):
    """Build this notebook's model, check the 5M budget, and return (model, n_trainable_params)."""
    options = resolve_architecture(architecture)
    model = init_weights(make_model(num_classes, architecture=options))
    # Plain attributes do not alter legacy state_dict keys or baseline repr/fingerprints.
    model._project_architecture = options
    model._project_description = describe_architecture(options)
    n_params = count_params(model)
    assert n_params <= PARAM_BUDGET, f"model {MODEL} has {n_params:,} trainable params > {PARAM_BUDGET:,}"
    return model, n_params


_model, N_PARAMS = build_model()
print(f"Model {MODEL}: {_model._project_description}")
print(f"Architecture options: {_model._project_architecture}")
print(f"Trainable parameters: {N_PARAMS:,} (budget {PARAM_BUDGET:,})")
assert N_PARAMS <= PARAM_BUDGET
assert N_PARAMS == expected_param_count(_model._project_architecture), "Unexpected architecture parameter count"
if SHOW_SUMMARY:
    try:
        from torchinfo import summary
    except ImportError:
        print("torchinfo is not installed; layer summary skipped (the parameter count above is complete)")
    else:
        print(summary(_model, input_size=(1, 3, 32, 32), depth=3, col_names=("output_size", "num_params"),
                      device="cpu", verbose=0))
del _model

## 5. Protocol: views, aggregation, McNemar, checkpoints

The next cell fixes the Q05 protocol (version 1, 2026-11-03) and defines the evaluation:

- **Views.** `q05_shift_u8` pads uint8 NHWC images by 2 px with the training crop's `crop_pad` (constant: black; reflect:
  mirror without the edge pixel) and cuts the 32 × 32 window at (2 + dy, 2 + dx); view pixel (i, j) is image pixel
  (i + dy, j + dx). Offset (0, 0) is the image itself. `q05_view_logits` normalizes each view with the checkpoint's
  statistics and flips flipped views after normalization, as `predict_logits` does; views (0, 0, orig/flip) are exactly
  the training notebooks' plain and flipped passes. All 18 views are computed once per image; the three sets are
  averages over their subsets.
- **Aggregation and test.** `q05_aggregate` averages softmax probabilities in float32 (for `flip2` bit for bit
  `0.5 * (plain + flipped)`); `q05_mcnemar` gives b, c and the exact two-sided binomial p-value.
- **Checkpoint and weights.** The run's recorded submission (latest `*_prediction-report.json`) decides best/last and
  raw/EMA, unless overridden; without one, the last checkpoint with EMA weights. The checkpoint file is read once, its
  SHA-256 compared with the export manifests' and prediction reports' values (a mismatch fails the run), and exactly
  those bytes are unpickled with the training notebooks' `load_checkpoint` (`weights_only=False`, as every notebook here:
  the checkpoints hold NumPy RNG state). Dataset fingerprint, run ID, architecture, model fingerprint, parameter count,
  strict state_dict keys and both split index arrays are checked before any evaluation.
- **Reproduction.** Stored accuracies of the same epoch and weights (prediction report and `*-validation.json`
  exports, which used this exact evaluation path) are compared with the reproduced plain and `flip2` accuracies;
  differences above `Q05_REPRO_TOLERANCE_PP` are flagged. The checkpoint's training-loop values (channels_last) are
  listed for information. The stored per-image probabilities (`*-validation.npz`) are compared image by image.

In [ ]:
# ---- Q05 shift TTA. The protocol constants below were fixed on 2026-11-03, before any Q05 result was seen; any
# change needs Q05_VERSION + 1 and a checklist note. All three view sets are reported for every model.
from fractions import Fraction

Q05_VERSION = 1
Q05_NAME = f"Q05_shift-tta-v{Q05_VERSION}"  # shared reports: result/reports/<MM-DD-HH-mm>_Q05_shift-tta-v1_<role>
Q05_ROLE = f"q05-shift-tta-v{Q05_VERSION}"  # per run: evaluations/validation/<RUN_ID>/<MM-DD-HH-mm>_<id>_q05-shift-tta-v1
Q05_SHIFT_PX = 2                            # padding width and largest window offset, in pixels
Q05_OFFSETS = {                             # (dy, dx) window offsets; each one is evaluated unflipped and flipped
    "flip2": ((0, 0),),                                                     # current inference: original + flip
    "shift5x2": ((0, 0), (0, -2), (0, 2), (-2, 0), (2, 0)),                 # centre, +-2 px horizontal / vertical
    "grid9x2": tuple((dy, dx) for dy in (-2, 0, 2) for dx in (-2, 0, 2)),   # 3 x 3 grid of offsets in {-2, 0, 2}^2
}
Q05_VIEW_SETS = tuple(Q05_OFFSETS)
Q05_BASELINE = "flip2"
Q05_COMPARISONS = (("shift5x2", "flip2"), ("grid9x2", "flip2"))  # exact paired McNemar for every model, all reported
Q05_TEST_SETS = ("shift5x2", "grid9x2")                          # optional test CSVs (flip2 = the existing CSVs)
Q05_PADDING_MODES = ("constant", "reflect")                      # the two training crop paddings (crop_pad)
Q05_PADDING_NOTES = {"constant": "zero (black) uint8 border before normalization, as the training random crop",
                     "reflect": "mirror image without the edge pixel, as the training random crop"}
# Registry aliases newer than the training notebooks' RUN_ID_ALIASES copy (Archive/run_identity_map.json): T115/T113
# launched as E10/E11 and are canonically E14/E15. canonical_run_id reads this dictionary at call time.
Q05_EXTRA_ALIASES = {"11-02-13-22_E_pyramid-ver-10-T115-ibn-a6s2": "11-02-13-22_E_pyramid-ver-14-T115-ibn-a6s2",
                     "11-02-13-23_E_pyramid-ver-11-T113-ibn-a6s1": "11-02-13-23_E_pyramid-ver-15-T113-ibn-a6s1"}
RUN_ID_ALIASES.update(Q05_EXTRA_ALIASES)
# E_trivialaug_ibn.ipynb (T118's notebook) saves four IBN options. With all of them off its network is exactly this
# notebook's basic E (its IBN placeholders are parameter-free nn.Identity modules), but its model fingerprint differs
# (its repr lists the placeholders). Pinned: that notebook's fingerprint of the IBN-off options below, recomputed with
# its own code by tests/test_q05_shift_tta.py. Active IBN-a/IBN-b options (T113, T115-T117) are not rebuilt here.
IBN_OPTION_KEYS = ("ibn_stage_blocks", "ibn_in_ratio", "ibn_b", "ibn_b_stem")
IBN_OFF_FINGERPRINTS = {
    "c2f5827a3eeb152dcd18e19a891a6a8fd8af2f41d651362d8cf75121083f3eca": {"implementation_version": 1, "drop_path": 0.1},
}


class Q05Skip(Exception):
    """Expected reason not to evaluate a listed run: no checkpoint, unsupported architecture or unfinished training."""


def q05_views(view_set):
    """(dy, dx, flip) of every view of a pre-declared set: each offset unflipped, then flipped."""
    return [(dy, dx, flip) for dy, dx in Q05_OFFSETS[view_set] for flip in (False, True)]


def q05_view_key(view):
    dy, dx, flip = view
    return f"dy{dy:+d}_dx{dx:+d}_{'flip' if flip else 'orig'}"


Q05_ALL_VIEWS = list(dict.fromkeys(view for name in Q05_VIEW_SETS for view in q05_views(name)))  # 18; flip2 first
if len(Q05_ALL_VIEWS) != 18 or not all((0, 0, False) in q05_views(n) and (0, 0, True) in q05_views(n)
                                       for n in Q05_VIEW_SETS):
    raise RuntimeError("Q05 view sets must contain the identity view and its flip, 18 distinct views in all")


def q05_shift_u8(x_u8, dy, dx, pad=Q05_SHIFT_PX, padding_mode="constant"):
    """The h x w window at offset (dy, dx) of uint8 NHWC images padded by `pad` px with the training crop's padding.

    Pixel (i, j) of the view is pixel (i + dy, j + dx) of the image (the content moves by (-dy, -dx)); outside the
    image it is the padding of crop_pad, the training notebooks' crop padding (section 3 copy): 0 = black in uint8
    before normalization for "constant", the mirror image without the edge pixel for "reflect". With pad <= 4 every
    view equals the training random crop's window at offset (4 + dy, 4 + dx). (0, 0) returns the images unchanged.
    """
    if padding_mode not in Q05_PADDING_MODES:
        raise ValueError(f"crop padding mode {padding_mode!r} is not one of {Q05_PADDING_MODES}")
    if (dy, dx) == (0, 0):
        return x_u8
    if max(abs(dy), abs(dx)) > pad:
        raise ValueError(f"offset ({dy}, {dx}) exceeds the {pad} px padding")
    h, w = x_u8.shape[1:3]
    padded = crop_pad(x_u8, pad, padding_mode)
    return padded[:, pad + dy:pad + dy + h, pad + dx:pad + dx + w]


@torch.inference_mode()
def q05_view_logits(model, images_u8, mean, std, device, views, padding_mode="constant", batch_size=1000,
                    channels_last=False):
    """{view: CPU float logits (N, 10)} for uint8 NHWC images and (dy, dx, flip) views, in eval mode.

    Each batch is copied to the device once; per view it is shifted in uint8 (q05_shift_u8), normalized with the
    checkpoint's statistics (to_float_nchw) and, for flipped views, flipped along the width after normalization,
    exactly as predict_logits: view (0, 0, False) is predict_logits(...) and (0, 0, True) predict_logits(..., flip=True)
    with the same batches, dtype (fp32, no autocast) and memory format.
    """
    model.eval()
    mean_t = torch.tensor(mean, device=device).view(1, 3, 1, 1)
    std_t = torch.tensor(std, device=device).view(1, 3, 1, 1)
    out = {view: [] for view in views}
    for i in range(0, len(images_u8), batch_size):
        xb = torch.as_tensor(np.asarray(images_u8[i:i + batch_size])).to(device)
        for view in views:
            dy, dx, flip = view
            x = to_float_nchw(q05_shift_u8(xb, dy, dx, Q05_SHIFT_PX, padding_mode), mean_t, std_t, channels_last)
            if flip:
                x = x.flip(3)
            out[view].append(model(x).float().cpu())
    return {view: torch.cat(chunks) for view, chunks in out.items()}


def q05_view_probabilities(model, images_u8, mean, std, device, padding_mode, batch_size, views=None):
    """{view: float32 softmax probabilities (N, 10)} of every Q05 view (F.softmax of the fp32 logits, as val_accuracy)."""
    logits = q05_view_logits(model, images_u8, mean, std, device, views or Q05_ALL_VIEWS, padding_mode, batch_size)
    return {view: F.softmax(value, 1).numpy() for view, value in logits.items()}


def q05_aggregate(view_probs, view_set):
    """Mean of a view set's softmax probabilities: float32 sum in view order, divided by the number of views.

    For flip2 this is bit for bit the training notebooks' 0.5 * (plain + flipped) (val_accuracy, predict_probs).
    """
    views = q05_views(view_set)
    total = np.array(view_probs[views[0]], dtype=np.float32, copy=True)
    for view in views[1:]:
        total += view_probs[view]
    return total / np.float32(len(views))


def q05_mcnemar(baseline_correct, new_correct):
    """Paired exact McNemar test of two predictions of the same images.

    b = baseline wrong and new right, c = baseline right and new wrong. Two-sided exact binomial p-value on the b + c
    discordant images with probability 1/2: p = min(1, 2 * P(X <= min(b, c))), X ~ Binomial(b + c, 1/2), computed in
    exact integer arithmetic; p = 1 when b + c = 0.
    """
    base = np.asarray(baseline_correct, dtype=bool)
    new = np.asarray(new_correct, dtype=bool)
    if base.ndim != 1 or base.shape != new.shape:
        raise ValueError("McNemar needs two equally long one-dimensional correctness vectors")
    b = int(np.count_nonzero(~base & new))
    c = int(np.count_nonzero(base & ~new))
    n = b + c
    tail = sum(math.comb(n, i) for i in range(min(b, c) + 1))
    p = 1.0 if n == 0 else float(min(Fraction(1), Fraction(2 * tail, 2 ** n)))
    return {"b": b, "c": c, "discordant": n, "p_value": p,
            "definition": "b = baseline wrong & new right, c = baseline right & new wrong; exact two-sided binomial"}


# ---- result roots (RESULT_DIR, then read-only extra roots) and a run's stored artifacts

def _q05_root_key(path):
    return os.path.normcase(os.path.abspath(os.path.expanduser(str(path))))


def q05_wait_for_folder(path, wait_s=0.0):
    """True once `path` is a folder. A Colab MyDrive shortcut can appear a few seconds after mounting: poll briefly."""
    deadline = time.monotonic() + max(0.0, float(wait_s))
    waiting = False
    while True:
        try:
            if path.is_dir():
                if waiting:
                    print(f"[q05] extra result root is now available: {path}")
                return True
        except OSError as error:  # e.g. a Drive mount that is still starting
            print(f"[q05] cannot access {path} yet: {error}")
        if time.monotonic() >= deadline:
            return False
        if not waiting:
            print(f"[q05] waiting up to {wait_s:g}s for {path} (a Drive shortcut may appear shortly after mounting)",
                  flush=True)
            waiting = True
        time.sleep(min(3.0, max(0.05, deadline - time.monotonic())))


_Q05_ROOTS = None


def q05_result_roots(refresh=False):
    """Search order for checkpoints and their artifacts: RESULT_DIR, then the available Q05_EXTRA_RESULT_ROOTS.

    Resolved once per session. Extra roots are only ever read; every Q05 output is written under RESULT_DIR.
    """
    global _Q05_ROOTS
    if _Q05_ROOTS is not None and not refresh:
        return list(_Q05_ROOTS)
    extras = Q05_EXTRA_RESULT_ROOTS or []
    if not isinstance(extras, (list, tuple)) or not all(isinstance(r, (str, Path)) and str(r) for r in extras):
        raise ValueError("Q05_EXTRA_RESULT_ROOTS must be a list of result folder paths (or empty)")
    own = Path(PATHS.result_dir)
    roots, seen = [own], {_q05_root_key(own)}
    for extra in map(Path, extras):
        if _q05_root_key(extra) in seen:
            continue
        seen.add(_q05_root_key(extra))
        if q05_wait_for_folder(extra, Q05_EXTRA_ROOT_WAIT_S if PATHS.env == "colab" else 0.0):
            roots.append(extra)
            print(f"[q05] extra result root (read-only): {extra}")
        else:
            print(f"[q05] WARNING: extra result root {extra} was not found; runs that exist only there are skipped. "
                  "On Colab check the MyDrive shortcut name (Drive > Shared with me > folder > Add shortcut to My "
                  "Drive), then rerun from section 1, or edit Q05_EXTRA_RESULT_ROOTS.", flush=True)
    _Q05_ROOTS = roots
    return list(roots)


def q05_root_kind(root):
    return "RESULT_DIR" if _q05_root_key(root) == _q05_root_key(PATHS.result_dir) else "extra (read-only)"


def q05_run_folders(roots, run_id, category):
    """Existing folders of one artifact category for a run (canonical and legacy names), across all roots in order.

    Every root is searched: this account's RESULT_DIR may hold only Q05/Q08 outputs for a teammate's run.
    """
    canonical = canonical_run_id(run_id)
    names = [canonical] + [legacy for legacy, target in RUN_ID_ALIASES.items() if target == canonical]
    found = []
    for root in roots:
        parent = getattr(run_paths(Path(root), canonical), category).parent
        found += [parent / name for name in names if (parent / name).is_dir()]
    return found


def _read_json(path):
    try:
        return json.loads(Path(path).read_text())
    except (OSError, ValueError):
        print(f"Warning: unreadable {path}; ignored")
        return None


def _number(value):
    return isinstance(value, (int, float)) and not isinstance(value, bool) and math.isfinite(value)


def q05_first_json(roots, run_id, category, pattern):
    """(data, path) of the first readable `pattern` file in the run's folders of `category`, or (None, None)."""
    for folder in q05_run_folders(roots, run_id, category):
        for path in sorted(folder.glob(pattern)):
            data = _read_json(path)
            if data:
                return data, path
    return None, None


def q05_recorded_submissions(roots, run_id):
    """Every prediction report (submission CSV) saved for the run, oldest first, deduplicated by file name."""
    found, seen = [], set()
    for folder in q05_run_folders(roots, run_id, "test"):
        for path in sorted(folder.glob("*_prediction-report*.json")):
            if path.name in seen:
                continue
            data = _read_json(path)
            if not data:
                continue
            seen.add(path.name)
            checkpoint = Path(str(data.get("checkpoint", ""))).name
            role = ("last" if checkpoint.endswith("_last-ckpt.pt") else
                    "best" if checkpoint.endswith("_best-ckpt.pt") else None)
            found.append({"report": path.name, "folder": str(folder), "checkpoint_role": role,
                          "checkpoint_file": checkpoint, "checkpoint_sha256": data.get("checkpoint_sha256"),
                          "epoch": data.get("epoch"), "weights": data.get("weights"), "tta": data.get("tta"),
                          "val": data.get("val") or {}, "submission": Path(str(data.get("submission", ""))).name,
                          "cache_fingerprint": data.get("cache_fingerprint")})
    return sorted(found, key=lambda s: s["report"])


def q05_request(entry):
    """{"run", "checkpoint", "weights"} of one RUN_IDS entry (a run ID or a dict with optional overrides)."""
    if isinstance(entry, str) and entry:
        return {"run": canonical_run_id(entry), "checkpoint": None, "weights": None}
    if isinstance(entry, dict) and set(entry) <= {"run", "checkpoint", "weights"} and isinstance(entry.get("run"), str):
        if entry.get("checkpoint") not in (None, "best", "last") or entry.get("weights") not in (None, "model", "ema"):
            raise ValueError(f"RUN_IDS entry {entry!r}: checkpoint must be best/last and weights model/ema")
        return {"run": canonical_run_id(entry["run"]), "checkpoint": entry.get("checkpoint"),
                "weights": entry.get("weights")}
    raise ValueError(f"RUN_IDS entries are run IDs or dicts with run/checkpoint/weights, got {entry!r}")


def q05_protocol(request, submissions):
    """Checkpoint role and weights: the request's override, else the run's recorded submission CSV (latest prediction
    report), else the pre-declared default (last checkpoint, EMA weights)."""
    recorded = [s for s in submissions if s["checkpoint_role"] in ("best", "last") and s["weights"] in ("model", "ema")]
    latest = recorded[-1] if recorded else None
    role = request["checkpoint"] or (latest["checkpoint_role"] if latest else "last")
    weights = request["weights"] or (latest["weights"] if latest else "ema")
    if request["checkpoint"] or request["weights"]:
        source = "override in RUN_IDS"
    elif latest:
        source = (f"recorded submission CSV ({latest['report']}"
                  + (f"; latest of {len(recorded)} reports" if len(recorded) > 1 else "") + ")")
    else:
        source = "no recorded submission: pre-declared default, last checkpoint with EMA weights"
    matching = [s for s in recorded if s["checkpoint_role"] == role and s["weights"] == weights and s["tta"]]
    return {"checkpoint": role, "weights": weights, "source": source,
            "submission": matching[-1] if matching else None}


def q05_label(run_id):
    match = re.search(r"_([A-Z])_[A-Za-z]+-ver-(\d+)-(T\d+)-", run_id)
    return f"{match.group(1)}{match.group(2)}/{match.group(3)}" if match else run_id


# ---- architecture support, checkpoint location and integrity

def q05_architecture_support(options, stored_fingerprint=None):
    """(canonical options, note) for a saved E architecture dictionary, or Q05Skip with the reason.

    Supported: every option of this notebook's E code (section 4: drop_path, shakedrop, block, depth, alpha), and an
    E_trivialaug_ibn.ipynb dictionary whose IBN options are all off and whose stored fingerprint is pinned
    (IBN_OFF_FINGERPRINTS), which is rebuilt as the identical basic E. note is None for natively supported options.
    """
    options = dict(ARCH_DEFAULTS) if options is None else options  # no saved options: E1's baseline (legacy)
    if not isinstance(options, dict):
        raise Q05Skip("saved architecture options are not a dictionary")
    ibn = {key: options[key] for key in IBN_OPTION_KEYS if key in options}
    canonical = {key: value for key, value in options.items() if key not in IBN_OPTION_KEYS}
    unknown = sorted(set(canonical) - set(ARCH_DEFAULTS) - set(ARCH_OPTION_DEFAULTS))
    if unknown:
        raise Q05Skip(f"unknown architecture options {unknown} (head256 / multistage heads such as T108/T109): "
                      "the E code copied from E_pyramid_t106_shakedrop.ipynb cannot rebuild them")
    note = None
    if ibn:
        blocks = ibn.get("ibn_stage_blocks", (0, 0, 0))
        if (not isinstance(blocks, (list, tuple)) or tuple(blocks) != (0, 0, 0)) or ibn.get("ibn_b", "off") != "off":
            raise Q05Skip(f"IBN-a/IBN-b options active {ibn} (T113, T115-T117): not rebuilt by this notebook's E "
                          "code (outside Q05's basic-E scope)")
        pinned = IBN_OFF_FINGERPRINTS.get(stored_fingerprint)
        try:
            same = pinned is not None and resolve_architecture(dict(pinned)) == resolve_architecture(dict(canonical))
        except ValueError:
            same = False
        if not same:
            raise Q05Skip(f"IBN options all off but the stored model fingerprint {str(stored_fingerprint)[:12]} is not "
                          "a pinned E_trivialaug_ibn.ipynb IBN-off fingerprint; architecture identity unverifiable")
        note = {"rule": "E_trivialaug_ibn.ipynb checkpoint with every IBN option off, rebuilt as the identical basic E "
                        "(stored fingerprint matched the pinned value; strict state_dict load)", "ibn_options": ibn}
    try:
        canonical = resolve_architecture(dict(canonical))
    except ValueError as error:
        raise Q05Skip(f"invalid architecture options {options}: {error}") from None
    return canonical, note


def q05_rebuild_view(ckpt):
    """(checkpoint seen through canonical options, note). Natively supported checkpoints are returned unchanged.

    For a pinned IBN-off checkpoint the IBN keys are dropped from the saved and resolved architecture (after checking
    that the rest equals the canonical basic E) and the fingerprint is replaced by the rebuilt model's, so that the
    verbatim validate_checkpoint and load_model (strict state_dict) apply unchanged. The checkpoint is not modified.
    """
    canonical, note = q05_architecture_support(ckpt["args"].get("architecture"), ckpt.get("model_fingerprint"))
    if note is None:
        return ckpt, None
    if "architecture" in ckpt:
        stored = {k: v for k, v in ckpt["architecture"].items() if k not in IBN_OPTION_KEYS}
        if json.loads(json.dumps(stored)) != json.loads(json.dumps(architecture_details(canonical))):
            raise ValueError("IBN-off checkpoint: resolved architecture differs from the canonical basic E")
    model, _ = build_model(architecture=canonical)
    view = dict(ckpt, args=dict(ckpt["args"], architecture=canonical), architecture=architecture_details(canonical),
                model_fingerprint=model_fingerprint(model))
    return view, dict(note, stored_fingerprint=ckpt.get("model_fingerprint"), canonical_options=canonical)


def q05_locate_checkpoint(run_id, role, roots):
    """(root, path) of the run's `role` checkpoint in the first root holding it (legacy names included)."""
    for root in roots:
        path = checkpoint_path(run_paths(Path(root), run_id).checkpoints, role)
        if path.is_file():
            return Path(root), path
    searched = ", ".join(str(run_paths(Path(root), run_id).checkpoints) for root in roots)
    raise Q05Skip(f"no {role} checkpoint under {searched or 'no result root'} (legacy names checked too); add the "
                  "result folder that holds it to Q05_EXTRA_RESULT_ROOTS")


def q05_recorded_checkpoint_hashes(roots, run_id, role, submissions):
    """SHA-256 values recorded for the run's `role` checkpoint: export manifests and prediction reports."""
    records, seen = [], set()
    suffix = f"_{role}-ckpt.pt"
    for folder in q05_run_folders(roots, run_id, "manifests"):
        for path in sorted(folder.glob("*_manifest*.json")):
            if path.name in seen:
                continue
            seen.add(path.name)
            for record in (_read_json(path) or {}).get("files", []):
                name = Path(str(record.get("path", ""))).name
                if str(record.get("path", "")).startswith("checkpoints/") and name.endswith(suffix) and record.get("sha256"):
                    records.append({"source": path.name, "file": name, "sha256": record["sha256"],
                                    "bytes": record.get("bytes")})
    for s in submissions:
        if s["checkpoint_role"] == role and s.get("checkpoint_sha256"):
            records.append({"source": s["report"], "file": s["checkpoint_file"], "sha256": s["checkpoint_sha256"],
                            "bytes": None})
    return records


def q05_integrity(sha256, size, records):
    values = sorted({r["sha256"] for r in records})
    if not values:
        return "unverified: no recorded SHA-256 for this checkpoint"
    if sha256 in values:
        sizes = {r["bytes"] for r in records if r["sha256"] == sha256 and r["bytes"] is not None}
        if sizes and sizes != {size}:
            raise ValueError(f"checkpoint size {size} differs from the recorded {sorted(sizes)} bytes")
        return ("verified against " + ", ".join(sorted({r['source'] for r in records if r['sha256'] == sha256}))
                + (" (other records list different hashes)" if len(values) > 1 else ""))
    message = f"checkpoint SHA-256 {sha256[:12]} differs from the recorded {[v[:12] for v in values]}"
    if not Q05_ALLOW_SHA256_MISMATCH:
        raise ValueError(message + "; set Q05_ALLOW_SHA256_MISMATCH=True to evaluate it anyway")
    print(f"Warning: {message} (allowed by Q05_ALLOW_SHA256_MISMATCH)")
    return "MISMATCH (allowed): " + message


# ---- stored accuracies and per-image probabilities (the reproduction check)

def q05_recorded_accuracies(roots, run_id, ckpt, weights, submissions):
    """Stored validation accuracies of this checkpoint epoch and weights: plain ("plain") and flip TTA ("flip2")."""
    refs = []
    metrics = ckpt.get("metrics") or {}
    key = "ema_val_acc" if weights == "ema" else "val_acc"
    if _number(metrics.get(key)):
        refs.append({"source": f"checkpoint metrics, epoch {ckpt['epoch']} (training-loop validation, channels_last)",
                     "kind": "checkpoint", "view_set": "plain", "accuracy_pct": float(metrics[key]), "npz": None})
    seen = set()
    for folder in q05_run_folders(roots, run_id, "validation"):
        for path in sorted(folder.glob("*-validation.json")):
            data = _read_json(path) if path.name not in seen else None
            if (not data or data.get("epoch") != ckpt["epoch"]
                    or data.get("cache_fingerprint", ckpt["cache_fingerprint"]) != ckpt["cache_fingerprint"]):
                continue
            seen.add(path.name)
            values = (data.get("metrics") or {}).get(weights) or {}
            npz = path.with_suffix(".npz")
            for mode, view_set in (("no_tta", "plain"), ("flip_tta", "flip2")):
                if _number(values.get(mode)):
                    refs.append({"source": path.name, "kind": "validation export", "view_set": view_set,
                                 "accuracy_pct": float(values[mode]), "npz": str(npz) if npz.is_file() else None})
    for s in submissions:
        values = (s["val"] or {}).get(weights) or {}
        if s["epoch"] != ckpt["epoch"] or s.get("cache_fingerprint", ckpt["cache_fingerprint"]) != ckpt["cache_fingerprint"]:
            continue
        for mode, view_set in (("no_tta", "plain"), ("flip_tta", "flip2")):
            if _number(values.get(mode)):
                refs.append({"source": s["report"], "kind": "prediction report", "view_set": view_set,
                             "accuracy_pct": float(values[mode]), "npz": None})
    return refs


def q05_reproduction(refs, reproduced, label):
    """Compare reproduced plain/flip2 accuracies with every stored value. Exports and prediction reports used this
    exact evaluation path: |difference| > Q05_REPRO_TOLERANCE_PP is flagged. The checkpoint's training-loop value
    (channels_last tensors) is listed for information only."""
    rows = []
    for ref in refs:
        value = reproduced[ref["view_set"]]
        diff = value - ref["accuracy_pct"]
        rows.append(dict(ref, reproduced_pct=value, difference_pp=diff,
                         flagged=bool(ref["kind"] != "checkpoint" and abs(diff) > Q05_REPRO_TOLERANCE_PP + 1e-9)))
    headline = next((r for kind in ("prediction report", "validation export") for r in rows
                     if r["kind"] == kind and r["view_set"] == "flip2"), None)
    if Q05_SMOKE:
        status = "not checked: smoke subset"
    elif headline is None:
        status = "no stored flip reference"
    elif headline["flagged"]:
        status = "MISMATCH"
    else:
        status = "reproduced"
    flagged = [r for r in rows if r["flagged"]] if not Q05_SMOKE else []
    if flagged:
        print(f"WARNING: {label}: stored accuracy not reproduced within {Q05_REPRO_TOLERANCE_PP}pp: " + "; ".join(
            f"{r['source']} {r['view_set']} {r['accuracy_pct']:.2f}% vs {r['reproduced_pct']:.2f}%" for r in flagged))
    return {"status": status, "tolerance_pp": Q05_REPRO_TOLERANCE_PP, "headline": headline, "references": rows,
            "flagged": len(flagged)}


def q05_compare_stored_probabilities(refs, weights, val_probs):
    """Per-image comparison with the first stored validation export (.npz) of this epoch: argmax agreement and the
    largest probability difference for plain and flip2."""
    path = next((Path(r["npz"]) for r in refs if r.get("npz")), None)
    if path is None:
        return {"status": "no stored validation .npz for this epoch"}
    with np.load(path, allow_pickle=False) as stored:
        indices = stored["indices"]
        if not np.array_equal(indices[:len(VAL_Y)], VAL_IDX[:len(VAL_Y)]):
            raise ValueError(f"{path.name}: stored validation indices differ from the Q05 split")
        out = {"status": "compared", "npz": path.name, "n": int(len(VAL_Y))}
        for name, key, probs in (("plain", f"{weights}_no_tta_probabilities", val_probs[(0, 0, False)]),
                                 ("flip2", f"{weights}_flip_tta_probabilities", q05_aggregate(val_probs, "flip2"))):
            if key not in stored.files:
                out[name] = {"status": f"{key} not stored"}
                continue
            ref = stored[key][:len(VAL_Y)]
            out[name] = {"argmax_agreement": int((ref.argmax(1) == probs.argmax(1)).sum()),
                         "max_abs_probability_difference": float(np.abs(ref.astype(np.float64) - probs).max())}
    return out


# ---- evaluation of one run

def q05_inference_settings(device):
    def flag(getter):
        try:
            return bool(getter())
        except Exception:  # absent in some PyTorch versions
            return None
    return {"precision": "fp32, no autocast (AMP not used), as the training notebooks' validation and prediction",
            "memory_format": "contiguous (channels_last off), as load_model + predict_logits in val_accuracy",
            "batch_size": EVAL_BATCH_SIZE, "device": device_name(device), "torch": torch.__version__,
            "cudnn_benchmark": bool(torch.backends.cudnn.benchmark),
            "cudnn_allow_tf32": flag(lambda: torch.backends.cudnn.allow_tf32),
            "cuda_matmul_allow_tf32": flag(lambda: torch.backends.cuda.matmul.allow_tf32),
            "aggregation": "mean of the views' softmax probabilities (float32), as predict_probs' flip TTA",
            "views": {name: [q05_view_key(v) for v in q05_views(name)] for name in Q05_VIEW_SETS},
            "shift_px": Q05_SHIFT_PX}


def q05_evaluate_run(entry, roots, device=None):
    """Evaluate one RUN_IDS entry on the 18 Q05 views. Returns (result, arrays); raises Q05Skip for expected skips."""
    device = device or DEV
    t_start = time.perf_counter()
    request = q05_request(entry)
    run_id = request["run"]
    label = q05_label(run_id)
    # Preflight from the saved configuration and progress: skip without reading the 60 MB checkpoint.
    config, config_path = q05_first_json(roots, run_id, "configs", "*_config.json")
    if config is not None:
        q05_architecture_support(config.get("args", {}).get("architecture"), config.get("model_fingerprint"))
    status, _ = q05_first_json(roots, run_id, "progress", "*_status.json")
    if status and status.get("status") != "completed" and not Q05_INCLUDE_INCOMPLETE:
        raise Q05Skip(f"training not completed (progress status {status.get('status')!r} at epoch "
                      f"{status.get('epoch')}); set Q05_INCLUDE_INCOMPLETE=True to evaluate its current checkpoint")
    submissions = q05_recorded_submissions(roots, run_id)
    protocol = q05_protocol(request, submissions)
    root, path = q05_locate_checkpoint(run_id, protocol["checkpoint"], roots)
    # Integrity: hash the bytes once and unpickle exactly those bytes (training notebooks' load_checkpoint,
    # torch.load(weights_only=False); their checkpoints hold NumPy RNG state that weights_only=True rejects).
    data = path.read_bytes()
    sha256 = hashlib.sha256(data).hexdigest()
    records = q05_recorded_checkpoint_hashes(roots, run_id, protocol["checkpoint"], submissions)
    integrity = q05_integrity(sha256, len(data), records)
    ckpt = load_checkpoint(io.BytesIO(data))
    del data
    if not isinstance(ckpt, dict) or "args" not in ckpt:
        raise ValueError(f"{path.name} is not a project training checkpoint")
    view, compat = q05_rebuild_view(ckpt)
    validate_checkpoint(view, META["fingerprint"], run_id=run_id)
    if not np.array_equal(ckpt["val_idx"].numpy(), VAL_IDX) or not np.array_equal(ckpt["train_idx"].numpy(), TRAIN_IDX):
        raise ValueError(f"{run_id}: checkpoint train/validation indices differ from the training notebooks' split")
    planned = ckpt["args"]["epochs"]
    # Completed: progress status, a final last checkpoint, or a recorded submission of this checkpoint (the training
    # notebooks write prediction reports only after training has finished).
    completed = (bool(status and status.get("status") == "completed")
                 or (protocol["checkpoint"] == "last" and ckpt["epoch"] >= planned)
                 or bool(protocol["submission"] and protocol["submission"]["epoch"] == ckpt["epoch"]))
    if not completed and protocol["checkpoint"] == "last" and not Q05_INCLUDE_INCOMPLETE:
        raise Q05Skip(f"training not completed (last checkpoint at epoch {ckpt['epoch']} of {planned})")
    weights = protocol["weights"]
    if weights == "ema" and ckpt.get("ema") is None:
        raise ValueError(f"{run_id}: the requested EMA weights are not in the checkpoint")
    options = saved_architecture(view["args"])
    model, n_params = load_model(view, weights, device)
    if n_params != expected_param_count(options) or n_params > PARAM_BUDGET:
        raise ValueError(f"{run_id}: parameter count {n_params:,} differs from the saved architecture's")
    padding_mode = ckpt["args"].get("crop_padding_mode", "constant")  # runs saved before the switch: black borders
    mean, std = ckpt["norm"]["mean"], ckpt["norm"]["std"]
    norm_diff = max(abs(a - b) for a, b in zip(list(mean) + list(std), list(TRAIN_NORM[0]) + list(TRAIN_NORM[1])))
    t_loaded = time.perf_counter()

    val_probs = q05_view_probabilities(model, VAL_X, mean, std, device, padding_mode, EVAL_BATCH_SIZE)
    set_probs = {name: q05_aggregate(val_probs, name) for name in Q05_VIEW_SETS}
    metrics = {name: classification_metrics(probs, VAL_Y) for name, probs in set_probs.items()}
    correct = {name: probs.argmax(1) == VAL_Y for name, probs in set_probs.items()}
    plain = classification_metrics(val_probs[(0, 0, False)], VAL_Y)
    single = {q05_view_key(v): float((p.argmax(1) == VAL_Y).mean() * 100) for v, p in val_probs.items()}
    t_val = time.perf_counter()
    refs = q05_recorded_accuracies(roots, run_id, ckpt, weights, submissions)
    reproduction = q05_reproduction(refs, {"plain": plain["accuracy_pct"], "flip2": metrics["flip2"]["accuracy_pct"]},
                                    f"{label} {protocol['checkpoint']}{ckpt['epoch']} {weights}")
    stored = q05_compare_stored_probabilities(refs, weights, val_probs)

    test_probs = None
    if TEST_X is not None:
        test_probs = q05_view_probabilities(model, TEST_X, mean, std, device, padding_mode, EVAL_BATCH_SIZE)
    t_test = time.perf_counter()
    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()

    view_sets = {}
    for name in Q05_VIEW_SETS:
        m = metrics[name]
        view_sets[name] = {"views": [q05_view_key(v) for v in q05_views(name)], "n_views": len(q05_views(name)),
                           "accuracy_pct": m["accuracy_pct"], "correct": m["correct"], "n": m["n"],
                           "cross_entropy": m["cross_entropy"], "per_class_recall": m["per_class_recall"],
                           "delta_vs_flip2_pp": m["accuracy_pct"] - metrics[Q05_BASELINE]["accuracy_pct"],
                           "mcnemar_vs_flip2": (q05_mcnemar(correct[Q05_BASELINE], correct[name])
                                                if name != Q05_BASELINE else None)}
    result = {
        "run_id": run_id, "label": label, "task_id": run_task_id(run_id), "q05": Q05_NAME,
        "smoke_subset": Q05_N if Q05_SMOKE else None,
        "protocol": {"checkpoint": protocol["checkpoint"], "weights": weights, "source": protocol["source"],
                     "epoch": ckpt["epoch"], "planned_epochs": planned, "training_completed": completed,
                     "recorded_submission": protocol["submission"]["report"] if protocol["submission"] else None},
        "checkpoint": {"file": path.name, "root": str(root), "root_kind": q05_root_kind(root), "sha256": sha256,
                       "integrity": integrity, "recorded": records,
                       "config": config_path.name if config_path else None,
                       "config_fingerprint_matches": (None if config is None else
                                                      config.get("model_fingerprint") == ckpt.get("model_fingerprint"))},
        "architecture": {"options": options, "description": describe_architecture(options), "n_params": n_params,
                         "stored_model_fingerprint": ckpt.get("model_fingerprint"), "compatibility": compat},
        "padding": {"mode": padding_mode, "pad_px": Q05_SHIFT_PX, "note": Q05_PADDING_NOTES[padding_mode],
                    "source": ("saved crop_padding_mode" if "crop_padding_mode" in ckpt["args"]
                               else "absent in the saved arguments: constant (black), the training default")},
        "normalization": {"mean": list(mean), "std": list(std), "source": "checkpoint (training split only)",
                          "max_abs_difference_vs_recomputed_training_split": norm_diff},
        "split": {"val_indices_sha256": int64_sha256(VAL_IDX), "matches_checkpoint": True, "n_validation": len(VAL_Y)},
        "inference": q05_inference_settings(device),
        "plain": {"accuracy_pct": plain["accuracy_pct"], "correct": plain["correct"]},
        "view_sets": view_sets, "single_view_accuracy_pct": single,
        "reproduction": reproduction, "stored_probabilities": stored,
        "timing_s": {"load": round(t_loaded - t_start, 2), "validation": round(t_val - t_loaded, 2),
                     "test": round(t_test - t_val, 2)},
    }
    arrays = {"val": val_probs, "test": test_probs, "submission": protocol["submission"]}
    return result, arrays


# ---- outputs (RESULT_DIR only; never overwritten)

def _q05_fresh(path):
    if Path(path).exists():
        raise FileExistsError(f"{path} exists; Q05 never overwrites files")
    return Path(path)


def q05_write_run(result, arrays, roots):
    """Per-run JSON (+ view probabilities .npz) under evaluations/validation/<RUN_ID>/ and, optionally, the shift5x2 /
    grid9x2 test CSVs (+ probabilities and a test report) under evaluations/test/<RUN_ID>/."""
    artifacts = run_paths(Path(PATHS.result_dir), result["run_id"])
    artifacts.validation.mkdir(parents=True, exist_ok=True)
    role = Q05_ROLE + ("-smoke" if Q05_SMOKE else "")
    path = artifact_path(artifacts, "validation", role, ".json", event=True, unique=True)
    result["test"] = q05_write_test(result, arrays, roots, artifacts)
    if Q05_SAVE_VIEW_PROBABILITIES:
        keys = [q05_view_key(v) for v in Q05_ALL_VIEWS]
        npz = _q05_fresh(path.with_suffix(".npz"))
        partial = npz.with_name(f".{npz.stem}.npz.{time.time_ns()}.tmp")
        try:
            with open(partial, "wb") as f:
                np.savez_compressed(f, labels=np.asarray(VAL_Y, dtype="<i8"),
                                    val_indices=np.asarray(VAL_IDX[:len(VAL_Y)], dtype="<i8"),
                                    view_keys=np.array(keys),
                                    probabilities=np.stack([arrays["val"][v] for v in Q05_ALL_VIEWS]).astype(np.float16))
            replace_file(partial, npz)
        finally:
            partial.unlink(missing_ok=True)
        result["view_probabilities_file"] = npz.name
    write_json_atomic(result, _q05_fresh(path))
    return path


def q05_write_test(result, arrays, roots, artifacts):
    """shift5x2 / grid9x2 test CSVs in the existing submission format (sample ID order, validate_submission), their
    probabilities and a test report; plus a read-only comparison of flip2 with the run's recorded CSV."""
    if arrays["test"] is None:
        return {"status": "not written: " + ("smoke subset" if Q05_SMOKE else "Q05_WRITE_TEST_CSVS is False")}
    sample_path = Path(CACHE_DIR) / "sample_submission.csv"
    sample = pd.read_csv(sample_path)
    row_of = {int(value): i for i, value in enumerate(TEST_IDS)}
    if len(row_of) != len(TEST_IDS) or set(sample["id"]) != set(row_of):
        raise ValueError("Test IDs differ from sample or contain duplicates")
    order = np.array([row_of[int(value)] for value in sample["id"]])
    flip2 = q05_aggregate(arrays["test"], "flip2")[order]
    out = {"flip2_vs_recorded_submission": q05_compare_recorded_csv(roots, result["run_id"], arrays["submission"],
                                                                     flip2, sample["id"].to_numpy())}
    status = result["reproduction"]["status"]
    if status not in ("reproduced", "no stored flip reference") or not result["protocol"]["training_completed"]:
        out["status"] = (f"not written: reproduction {status}" if status not in ("reproduced", "no stored flip reference")
                         else "not written: training not completed")
        print(f"[q05] {result['label']}: test CSVs {out['status']}")
        return out
    artifacts.test.mkdir(parents=True, exist_ok=True)
    out["status"], out["files"] = "written", {}
    for name in Q05_TEST_SETS:
        ordered = q05_aggregate(arrays["test"], name)[order]
        csv_path = artifact_path(artifacts, "test", f"q05-{name}-submission", ".csv", event=True, unique=True)
        probs_path = _q05_fresh(csv_path.with_name(csv_path.stem.replace("-submission", "-probabilities") + ".npz"))
        report_path = _q05_fresh(csv_path.with_name(csv_path.stem.replace("-submission", "-test-report") + ".json"))
        pd.DataFrame({"id": sample["id"], "label": ordered.argmax(1).astype(np.int64)}).to_csv(_q05_fresh(csv_path),
                                                                                                index=False)
        rows = validate_submission(csv_path, sample_path)
        np.savez_compressed(probs_path, ids=sample["id"].to_numpy(), probabilities=ordered)
        vs = result["view_sets"][name]
        write_json_atomic({
            "run_id": result["run_id"], "q05": Q05_NAME, "view_set": name, "views": vs["views"],
            "padding": result["padding"], "checkpoint": result["checkpoint"]["file"],
            "checkpoint_sha256": result["checkpoint"]["sha256"], "epoch": result["protocol"]["epoch"],
            "weights": result["protocol"]["weights"], "aggregation": "mean of the views' softmax probabilities",
            "validation_accuracy_pct": vs["accuracy_pct"], "validation_flip2_accuracy_pct":
                result["view_sets"]["flip2"]["accuracy_pct"], "validation_delta_vs_flip2_pp": vs["delta_vs_flip2_pp"],
            "submission": csv_path.name, "rows": rows, "format_check": "passed",
            "changed_labels_vs_flip2": int((ordered.argmax(1) != flip2.argmax(1)).sum()),
            "cache_fingerprint": META["fingerprint"], "kaggle_public_accuracy": None,
            "note": "Generated by Q05_shift_tta.ipynb; not submitted. Any Kaggle submission needs the user's approval."},
            report_path)
        out["files"][name] = {"csv": csv_path.name, "probabilities": probs_path.name, "report": report_path.name,
                              "rows": rows, "changed_labels_vs_flip2": int((ordered.argmax(1) != flip2.argmax(1)).sum())}
    return out


def q05_compare_recorded_csv(roots, run_id, submission, flip2_ordered, sample_ids):
    """Read-only check of the test path: flip2 labels against the run's recorded submission CSV of this protocol."""
    if submission is None or not submission.get("submission"):
        return {"status": "no recorded submission CSV for this checkpoint/weights"}
    csv_path = next((folder / submission["submission"] for folder in q05_run_folders(roots, run_id, "test")
                     if (folder / submission["submission"]).is_file()), None)
    if csv_path is None:
        return {"status": f"recorded CSV {submission['submission']} not found"}
    recorded = pd.read_csv(csv_path)
    if not np.array_equal(recorded["id"].to_numpy(), sample_ids):
        return {"status": "recorded CSV IDs differ from the sample order", "csv": csv_path.name}
    out = {"status": "compared", "csv": csv_path.name, "n": int(len(recorded)),
           "label_agreement": int((recorded["label"].to_numpy() == flip2_ordered.argmax(1)).sum())}
    probs_path = csv_path.with_name(csv_path.stem.replace("_submission", "_probabilities") + ".npz")
    if probs_path.is_file():
        with np.load(probs_path, allow_pickle=False) as stored:
            if np.array_equal(stored["ids"], sample_ids):
                out["max_abs_probability_difference"] = float(
                    np.abs(stored["probabilities"].astype(np.float64) - flip2_ordered).max())
    return out


def q05_event_path(folder, role, extension):
    """Dated shared Q05 artifact MM-DD-HH-mm_Q05_shift-tta-v<k>_<role><ext>, numbered on collision."""
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    stem = f"{run_timestamp()}_{Q05_NAME}_{role}"
    path, index = folder / (stem + extension), 1
    while any(folder.glob(path.stem + "*")):
        index += 1
        path = folder / f"{stem}-{index}{extension}"
    return path


def q05_summary_rows(results):
    """One row per evaluated run and view set."""
    rows = []
    for r in results:
        repro = r["reproduction"]
        headline = repro["headline"] or {}
        for name in Q05_VIEW_SETS:
            vs = r["view_sets"][name]
            test = (r.get("test") or {}).get("files", {}).get(name, {})
            mc = vs["mcnemar_vs_flip2"] or {}
            rows.append({"run_id": r["run_id"], "label": r["label"], "checkpoint": r["protocol"]["checkpoint"],
                         "epoch": r["protocol"]["epoch"], "weights": r["protocol"]["weights"],
                         "padding_mode": r["padding"]["mode"], "view_set": name, "n_views": vs["n_views"],
                         "accuracy_pct": vs["accuracy_pct"], "correct": vs["correct"], "n": vs["n"],
                         "delta_vs_flip2_pp": vs["delta_vs_flip2_pp"], "mcnemar_b": mc.get("b"),
                         "mcnemar_c": mc.get("c"), "mcnemar_p": mc.get("p_value"), "cross_entropy": vs["cross_entropy"],
                         "recorded_flip2_pct": headline.get("accuracy_pct"), "recorded_source": headline.get("source"),
                         "reproduction": repro["status"], "flagged_references": repro["flagged"],
                         "integrity": r["checkpoint"]["integrity"].split(":")[0].split(" against")[0],
                         "checkpoint_root_kind": r["checkpoint"]["root_kind"],
                         "test_csv": test.get("csv"), "test_changed_labels_vs_flip2": test.get("changed_labels_vs_flip2")})
    return rows

## 6. Evaluate the runs

For each run in `RUN_IDS`: preflight (saved configuration and progress), checkpoint location and integrity, identity
checks, the 18 validation views, the three view sets, the McNemar comparisons, the reproduction check and, with
`Q05_WRITE_TEST_CSVS=True`, the 18 test views and two new CSVs (only for completed runs whose `flip2` reproduced or has
no stored reference). The flip2 test labels are also compared with the run's recorded submission CSV (read only).
The printed table lists every evaluated model; skipped and failed runs follow with their reasons. Results are
diagnostics on one fixed split; a positive Δ is not by itself evidence of a better hidden-test score, and any CSV
submission needs the user's approval.

In [ ]:
if DEV.type == "cuda":
    torch.backends.cudnn.benchmark = bool(HW.cudnn_benchmark)
Q05_RESULTS, Q05_SKIPPED, Q05_FAILED = {}, {}, {}
_t0 = time.perf_counter()
_roots = q05_result_roots() if RUN_IDS else []
if _roots:
    print("[q05] checkpoint search order: " + " -> ".join(f"{r} ({q05_root_kind(r)})" for r in _roots)
          + f"; outputs go to {PATHS.result_dir} only")
_settings = q05_inference_settings(DEV)
print(f"[q05] inference: {_settings['precision']}; {_settings['memory_format']}; batch {EVAL_BATCH_SIZE}; "
      f"{_settings['device']}; cudnn.benchmark {_on(_settings['cudnn_benchmark'])}; TF32 conv "
      f"{_settings['cudnn_allow_tf32']} / matmul {_settings['cuda_matmul_allow_tf32']} (PyTorch defaults, unchanged)")
print("[q05] pre-declared view sets: " + "; ".join(
    f"{name} ({len(q05_views(name))} views: offsets {list(Q05_OFFSETS[name])} x orig/flip)" for name in Q05_VIEW_SETS))
for _entry in RUN_IDS:
    _name = _entry if isinstance(_entry, str) else _entry.get("run") if isinstance(_entry, dict) else _entry
    print(f"[q05] {q05_label(str(_name))}: {_name}", flush=True)
    try:
        _result, _arrays = q05_evaluate_run(_entry, _roots)
        _path = q05_write_run(_result, _arrays, _roots)
    except Q05Skip as _skip:
        Q05_SKIPPED[str(_name)] = str(_skip)
        print(f"[q05]   SKIPPED: {_skip}", flush=True)
        continue
    except Exception as _error:  # identity, split, integrity or I/O problems: recorded, then the next run
        if Q05_STOP_ON_ERROR:
            raise
        Q05_FAILED[str(_name)] = f"{type(_error).__name__}: {_error}"
        print(f"[q05]   FAILED: {type(_error).__name__}: {_error}", flush=True)
        continue
    Q05_RESULTS[_result["run_id"]] = _result
    _vs, _rep = _result["view_sets"], _result["reproduction"]
    _head = _rep["headline"]
    print(f"[q05]   {_result['protocol']['checkpoint']} epoch {_result['protocol']['epoch']} {_result['protocol']['weights']} "
          f"({_result['protocol']['source']}); padding {_result['padding']['mode']}; checkpoint from "
          f"{_result['checkpoint']['root_kind']}; {_result['checkpoint']['integrity']}")
    print(f"[q05]   flip2 {_vs['flip2']['accuracy_pct']:.2f}% (recorded "
          + (f"{_head['accuracy_pct']:.2f}%, {_head['difference_pp']:+.2f}pp" if _head else "none")
          + f": {_rep['status']}" + (f"; {_rep['flagged']} reference(s) flagged" if _rep["flagged"] else "") + ") | "
          + " | ".join(f"{n} {_vs[n]['accuracy_pct']:.2f}% ({_vs[n]['delta_vs_flip2_pp']:+.2f}pp, "
                       f"b {_vs[n]['mcnemar_vs_flip2']['b']} c {_vs[n]['mcnemar_vs_flip2']['c']} "
                       f"p {_vs[n]['mcnemar_vs_flip2']['p_value']:.3g})" for n in Q05_VIEW_SETS if n != Q05_BASELINE)
          + f" | {sum(_result['timing_s'].values()):.0f}s -> {_path.name}", flush=True)
    del _arrays

Q05_ELAPSED_S = round(time.perf_counter() - _t0, 1)
_role = "summary-smoke" if Q05_SMOKE else "summary"
Q05_SUMMARY_PATH = q05_event_path(Path(PATHS.result_dir) / "reports", _role, ".csv")
Q05_REPORT_PATH = _q05_fresh(Q05_SUMMARY_PATH.with_name(
    Q05_SUMMARY_PATH.stem.replace(f"_{_role}", f"_{_role.replace('summary', 'report')}") + ".json"))
Q05_SUMMARY = pd.DataFrame(q05_summary_rows(Q05_RESULTS.values()))
Q05_SUMMARY.to_csv(_q05_fresh(Q05_SUMMARY_PATH), index=False)
write_json_atomic({
    "q05": Q05_NAME, "created_at": datetime.now(ZoneInfo("America/New_York")).isoformat(timespec="seconds"),
    "pre_declared": {"date": "2026-11-03", "view_sets": {n: [q05_view_key(v) for v in q05_views(n)] for n in Q05_VIEW_SETS},
                     "offsets": {n: [list(o) for o in Q05_OFFSETS[n]] for n in Q05_VIEW_SETS},
                     "shift_px": Q05_SHIFT_PX, "padding": "each run's training crop padding (crop_padding_mode)",
                     "aggregation": "mean of softmax probabilities over views",
                     "comparisons": [{"new": a, "baseline": b, "test": "exact two-sided McNemar"} for a, b in Q05_COMPARISONS],
                     "selection": "none: every view set is reported for every model"},
    "inference": _settings, "split": {"val_indices_sha256": int64_sha256(VAL_IDX), "n_validation": len(VAL_Y),
                                      "smoke_subset": Q05_N if Q05_SMOKE else None, "cache_fingerprint": META["fingerprint"]},
    "roots": [{"root": str(r), "kind": q05_root_kind(r)} for r in _roots], "run_ids": [str(e) for e in RUN_IDS],
    "evaluated": list(Q05_RESULTS), "skipped": Q05_SKIPPED, "failed": Q05_FAILED, "elapsed_s": Q05_ELAPSED_S,
    "summary_csv": Q05_SUMMARY_PATH.name, "results": list(Q05_RESULTS.values())}, Q05_REPORT_PATH)
print(f"\n[q05] {len(Q05_RESULTS)} evaluated, {len(Q05_SKIPPED)} skipped, {len(Q05_FAILED)} failed in {Q05_ELAPSED_S:.0f}s; "
      f"summary {Q05_SUMMARY_PATH}; report {Q05_REPORT_PATH.name}")
if len(Q05_SUMMARY):
    _key = "label" if Q05_SUMMARY.groupby("label")["run_id"].nunique().max() == 1 else "run_id"
    _wide = Q05_SUMMARY.pivot(index=_key, columns="view_set", values="accuracy_pct")[list(Q05_VIEW_SETS)]
    _first = Q05_SUMMARY.drop_duplicates(_key).set_index(_key)
    _table = pd.DataFrame({
        "protocol": _first["checkpoint"] + _first["epoch"].astype(str) + " " + _first["weights"],
        "pad": _first["padding_mode"], "recorded_flip2": _first["recorded_flip2_pct"], "flip2": _wide["flip2"],
        "repro": _first["reproduction"]})
    for _n in Q05_VIEW_SETS:
        if _n == Q05_BASELINE:
            continue
        _rows = Q05_SUMMARY[Q05_SUMMARY["view_set"] == _n].set_index(_key)
        _table[_n] = _wide[_n]
        _table[f"d_{_n}_pp"] = _rows["delta_vs_flip2_pp"]
        _table[f"b/c_{_n}"] = _rows["mcnemar_b"].astype(int).astype(str) + "/" + _rows["mcnemar_c"].astype(int).astype(str)
        _table[f"p_{_n}"] = _rows["mcnemar_p"]
    with pd.option_context("display.width", 250, "display.max_columns", None, "display.precision", 3):
        print(_table.loc[[r[_key] for r in Q05_RESULTS.values()]].to_string())
    print("Accuracy in % on the fixed 5k validation split; d = view set minus flip2 (pp); b = flip2 wrong & new right, "
          "c = flip2 right & new wrong; p = exact two-sided McNemar, unadjusted, one test per model and view set.")
for _name, _reason in Q05_SKIPPED.items():
    print(f"SKIPPED {q05_label(_name)}: {_reason}")
for _name, _reason in Q05_FAILED.items():
    print(f"FAILED {q05_label(_name)}: {_reason}")
_flagged = [r["label"] for r in Q05_RESULTS.values() if r["reproduction"]["status"] == "MISMATCH" or r["reproduction"]["flagged"]]
if _flagged:
    print(f"WARNING: stored accuracies not reproduced within {Q05_REPRO_TOLERANCE_PP}pp for {_flagged}; see the report")